In [ ]:
import simpy
import pandas as pd 
import numpy as np
import random
import math                         # [수정사항] 유클리드 대각선 거리를 계산하기 위해 math 모듈 추가
from itertools import combinations  # 주문 내에서 함께 출고되는 부품쌍 생성 (Kitting Slotting에 사용)
from collections import Counter     # 함께 출고되는 부품쌍의 출현 빈도 계산 (어떤 부품을 서로 가깝게 배치할지 판단하기 위함)
import matplotlib.pyplot as plt     # 시각화 및 스타일링
import seaborn as sns               # 히트맵용 시각화
import matplotlib.patches as patches # 2D 궤적 맵에서 랙(Rack)을 그리기 위함

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# --- 시스템 환경 설정 (상수 정의) ---
# 창고의 물리적 레이아웃과 물류 로봇 성능 정의
RACK_START_Y, RACK_END_Y = 4.0, 15.1       # 랙이 설치된 Y축의 시작과 끝 좌표 범위 (4.0m ~ 15.1m 사이에만 적재 가능)
AISLE_X = [1.2, 3.6, 6.0, 8.4, 10.8, 13.2] # 창고 통로(Aisle)가 위치한 X축 좌표 리스트

# 작업 스테이션 위치 (x, y, z): 레이아웃과 일치하도록 S01은 A02(3.6)와, S02는 A05(10.8)와 X좌표를 완벽히 맞춤
STATIONS = {'S01': [3.6, 0.5, 0], 'S02': [10.8, 0.5, 0]} 

CTU_SPEED_XY = 1.8  # 로봇의 XY 이동 속도 (m/s)
CTU_SPEED_Z = 0.5   # 로봇의 리프트(Z) 상하 이동 속도 (m/s)
TOTE_CAPACITY = 7   # CTU가 한 번에 운반할 수 있는 최대 토트 박스 수
SHELF_HEIGHTS = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0] # 선반의 층별 높이(Z축) 리스트 (부품의 Z좌표로 사용)

# 교차 통로(Cross Aisle) 설정
# 로봇이 다른 통로로 넘어갈 때 랙을 관통하지 않고 앞쪽으로 빠져나와서 이동하는 기준 Y 좌표 
CROSS_AISLE_Y = 3.0

# ==========================================
# 창고 레이아웃 기반 예상 소요 시간 산출 함수 (기존 거리 함수 대체)
# ==========================================
def get_estimated_travel_time(pos1, pos2): # [수정사항: 1번 이슈 해결] 단순 거리에서 속도가 반영된 소요 시간 계산으로 변경
    """
    물류 로봇이 랙(Rack)을 뚫고 지나갈 수 없다는 물리적 제약을 반영하여
    두 위치(pos1, pos2) 사이의 '예상 이동 소요 시간'을 계산하는 함수.
    디스패칭 시 어떤 작업이 가장 '빨리' 도달할 수 있는지 판단하는 기준이 됨.
    """
    x1, y1, z1 = pos1 # 시작 지점의 x, y, z 좌표를 언패킹
    x2, y2, z2 = pos2 # 도착 지점의 x, y, z 좌표를 언패킹
    
    # 1. 같은 통로(X좌표 동일)일 경우: 단순히 Y축 깊이 차이만 계산
    if x1 == x2: # 시작점과 도착점의 X 좌표(통로)가 같은지 확인
        xy_dist = abs(y1 - y2) # 같은 통로면 Y 좌표의 차이 절대값을 평면 거리로 산출
    # 2. 다른 통로일 경우: 현재 통로에서 교차 통로(Y=3.0)로 직진하여 빠져나온 뒤, 가로(X)로 이동 후 새 통로로 진입
    else: # 다른 통로로 이동해야 하는 경우
        xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y) # 현재 Y에서 교차통로로 나가는 거리 + X축 이동 거리 + 새 통로로 들어가는 거리 합산
        
    dz = abs(z1 - z2) # Z축(높이) 차이 계산
    
    # [수정사항: 1번 이슈 해결] 평면 이동 속도와 리프트 속도 중 병목이 되는(더 오래 걸리는) 진짜 소요 시간을 반환
    return max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z) 

# ==========================================
# CTU class 물류 로봇
# ==========================================
class CTU: # 물류 로봇(CTU) 객체를 정의하는 클래스
    def __init__(self, env, id, warehouse, dispatch_rule): # 로봇 초기화 메서드 (환경, ID, 소속 창고, 디스패칭 룰을 인자로 받음)
        self.env = env # SimPy 시뮬레이션 환경 객체를 로봇의 속성으로 저장
        self.id = id # CTU를 구별하기 위한 고유 번호(CTU 0 또는 1)
        self.warehouse = warehouse # 로봇이 작업할 창고 객체의 참조를 저장
        self.dispatch_rule = dispatch_rule # 현재 로봇에 적용할 작업 할당 방식(Nearest, Density 등) 저장
        
        # 초기 로봇 위치 설정: 0번 CTU는 S01에서, 1번 CTU는 S02에서 시작
        self.pos = STATIONS['S01'] if id == 0 else STATIONS['S02'] # ID가 0이면 S01 좌표, 아니면 S02 좌표를 초기 위치로 설정

        # 총 이동거리 및 작업 수 기록
        self.total_distance = 0 # 로봇이 이동한 총 누적 거리를 0으로 초기화
        self.completed_jobs = 0 # 로봇이 완료한 총 피킹 작업 수를 0으로 초기화
        self.busy = False # 로봇의 현재 상태를 대기 중(False)으로 초기화
        
        # 이 CTU가 현재 작업하기 위해 점유 중인 통로(X좌표)들을 담아두는 집합(Set) 변수 추가
        self.current_batch_aisles = set() 
        
        self.process = env.process(self.run()) # 로봇의 메인 작업 루프인 run()을 SimPy 이벤트 큐에 프로세스로 등록

    # 이동 시간 및 거리 계산 (맨해튼 직교 거리 기반)
    def calculate_travel_time(self, target_pos, is_return=False): # [수정사항] 복귀 여부를 알 수 있는 is_return 파라미터 기본값 False로 추가
        x1, y1, z1 = self.pos # 로봇의 현재 위치 좌표 언패킹
        x2, y2, z2 = target_pos # 로봇이 이동해야 할 목표 위치 좌표 언패킹
        
        if is_return: 
            # [수정사항] 스테이션으로 복귀할 경우: 
            # 랙 내부(Y>=4.0)에서 교차 통로(Y=3.0)까지는 통로를 따라 직진(Manhattan)으로 빠져나오고,
            # Y=3.0 교차 통로에서부터 스테이션(Y=0.5) 구간은 장애물이 없는 열린 공간이므로 대각선(유클리드)으로 가로지릅니다.
            dist_to_cross_aisle = abs(y1 - CROSS_AISLE_Y) 
            euclidean_dist_to_station = math.sqrt((x1 - x2)**2 + (CROSS_AISLE_Y - y2)**2)
            xy_dist = dist_to_cross_aisle + euclidean_dist_to_station
        else:
            if x1 == x2: # 출발지와 목적지의 통로(X좌표)가 동일한지 확인
                xy_dist = abs(y1 - y2) # 통로가 같으면 랙 안쪽으로 직진하는 거리만 계산
            else: # 통로가 다른 경우
                xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y) # 앞쪽 교차 통로(Y=3.0)로 우회하는 평면 궤적 거리 계산
            
        dz = abs(z1 - z2) # Z축(상하 리프트) 이동 거리 계산
        
        travel_time = max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z) # 평면 주행과 리프트 승강은 동시 수행되므로 둘 중 더 오래 걸리는 시간을 소요 시간으로 산정
        travel_distance = xy_dist + dz # 물리적인 총 모터 이동 거리(주행 거리 + 리프트 이동 거리) 합산
        
        return travel_time, travel_distance # 계산된 이동 시간과 총 이동 거리를 튜플로 반환

    # CTU의 실제 작업 수행 프로세스
    def run(self): # 시간 흐름에 따라 CTU가 지속적으로 수행할 제너레이터 메서드
        while True: # 시뮬레이션 종료 시까지 무한 반복
            if self.warehouse.job_queue: # 창고에 처리 대기 중인 주문 작업이 남아있는지 확인
                # 무조건 busy를 True로 바꾸지 않고, 겹치지 않는 배치를 받아올 수 일 때만 상태를 변경하도록 로직 순서 변경
                batch = self.warehouse.apply_dispatching(self, self.dispatch_rule) 
                
                if batch: # 겹치지 않는 할당받은 작업 묶음이 존재한다면
                    self.busy = True # 비로소 작업 상태(True)로 전환
                    
                    # 할당받은 작업 배치에 포함된 모든 통로의 X좌표를 추출하여 현재 CTU의 점유 통로로 등록
                    # (이후 다른 CTU가 디스패칭을 수행할 때 이 통로의 작업들은 투명 인간처럼 무시됨)
                    self.current_batch_aisles = set(job['pos'][0] for job in batch)
                    
                    # [수정사항: 1번 이슈 해결] 거리 기준 정렬에서 '예상 소요 시간' 기준 정렬로 변경
                    batch.sort(key=lambda j: get_estimated_travel_time(self.pos, j['pos'])) 
                    
                    for job in batch: # 할당받은 각 작업(부품 피킹)을 순회
                        t, d = self.calculate_travel_time(job['pos']) # 해당 부품 위치까지의 이동 시간(t)과 거리(d)를 계산
                        
                        self.warehouse.move_log.append({ # 분석을 위해 로봇의 이동 상세 이력을 창고의 로그 리스트에 추가
                            'time': self.env.now, # 현재 시뮬레이션 시간 기록
                            'ctu_id': self.id, # 어떤 로봇이 이동했는지 ID 기록
                            'from_x': self.pos[0], 'from_y': self.pos[1], 'from_z': self.pos[2], # 출발 지점 X, Y, Z 좌표 기록
                            'to_x': job['pos'][0], 'to_y': job['pos'][1], 'to_z': job['pos'][2], # 도착(피킹) 지점 X, Y, Z 좌표 기록
                            'item_id': job['item_id'], # 피킹한 대상 부품 코드 기록
                            'travel_time': t, 'travel_distance': d, # 해당 구간 이동 소요 시간과 이동 거리 기록
                            'dispatch_rule': self.dispatch_rule, # 당시 적용된 디스패칭 룰 기록
                        })
                        
                        yield self.env.timeout(t) # 계산된 이동 시간(t)만큼 시뮬레이션 시간을 흘려보냄 
                        
                        self.pos = job['pos'] # 로봇의 현재 위치를 방금 피킹을 완료한 목적지 좌표로 갱신
                        self.total_distance += d # 로봇의 누적 이동 거리에 방금 이동한 거리를 더함
                        self.completed_jobs += 1 # 완료한 피킹 작업 개수 1 증가
                        
                    # 마지막 피킹 위치의 X좌표에 따라 복귀할 지정 스테이션 결정
                    last_job_x = self.pos[0] # 배치의 모든 피킹이 끝난 후 로봇의 현재(마지막) 통로 X 좌표 확인
                    target_station = STATIONS['S01'] if last_job_x <= 6.0 else STATIONS['S02'] # 마지막 X 좌표가 6.0 이하(A01~A03)면 S01, 초과면 S02를 복귀 지점으로 결정
                    
                    # [수정사항] 복귀할 때는 is_return=True를 넘겨주어 Y=3.0 앞쪽에서는 유클리드 거리로 대각선 복귀하도록 호출
                    t, d = self.calculate_travel_time(target_station, is_return=True) 
                    
                    self.warehouse.move_log.append({ # 분석을 위해 로봇의 스테이션 복귀 이력을 창고 로그에 추가
                        'time': self.env.now, # 현재 시뮬레이션 시간 기록
                        'ctu_id': self.id, # 복귀하는 로봇 ID 기록
                        'from_x': self.pos[0], 'from_y': self.pos[1], 'from_z': self.pos[2], # 마지막 피킹 지점 좌표 기록
                        'to_x': target_station[0], 'to_y': target_station[1], 'to_z': target_station[2], # 목적지 스테이션 좌표 기록
                        'item_id': 'RETURN', # 피킹 작업이 아닌 스테이션 복귀임을 의미하는 태그 삽입
                        'travel_time': t, 'travel_distance': d, # 복귀 구간 소요 시간 및 거리 기록
                        'dispatch_rule': self.dispatch_rule, # 당시 적용된 디스패칭 룰 기록
                    })
                    
                    yield self.env.timeout(t) # 복귀 시간만큼 시뮬레이션 시간 지연
                    self.pos = target_station # 복귀가 완료되었으므로 로봇 위치를 스테이션 좌표로 갱신
                    self.total_distance += d # 복귀 이동 거리 누적 더하기
                
                    # 모든 작업과 복귀를 마쳤으므로 점유했던 통로들을 깨끗하게 비워 해제
                    self.current_batch_aisles.clear()
                    self.busy = False # 로봇 상태를 다시 대기 중(False)으로 변경
                else: 
                    # 큐에 짐은 남아있으나, 다른 CTU가 모두 점유하고 있어 배치를 받지 못한 경우 1초간 대기하며 눈치보기
                    yield self.env.timeout(1) 
            else: # 아예 창고 큐 전체에 남은 할당 작업이 없는 경우
                yield self.env.timeout(1) # 무한 루프 과부하를 막기 위해 1초 대기 후 다시 작업 확인


# ==========================================
# 창고 시스템 클래스 정의
# ==========================================
class Warehouse: # 전체 창고, 재고, 큐, 로그를 통합 관리하는 클래스
    def __init__(self, env, inventory, plan_df): # 창고 초기화 메서드
        self.env = env # SimPy 시뮬레이션 환경 저장
        self.inventory = inventory # 슬로팅 룰에 의해 생성된 {부품: 위치} 맵핑 데이터 저장
        self.plan_df = plan_df # 금일 처리해야 할 전체 출고 계획 주문 데이터 저장
        self.job_queue = [] # 실시간으로 쏟아지는 주문(부품 피킹)들이 대기하는 리스트(큐) 초기화
        
        self.dispatch_log = [] # 디스패칭 룰에 의해 배치 작업이 어떻게 생성되었는지 기록할 리스트
        self.move_log = [] # 로봇의 점대점 이동 상세 이력을 기록할 리스트
        
        self.ctus = [] # 교착(Deadlock) 방지를 위해 창고가 전체 CTU들의 객체 정보를 알고 있도록 리스트 초기화

    def apply_dispatching(self, ctu, rule): # 룰에 따라 큐에서 작업을 꺼내 로봇에 할당해주는 메서드
        if not self.job_queue: return [] # 만약 큐가 비어있다면 빈 리스트 반환하여 작업 할당 취소

        # 병목 현상 방지: 현재 작업을 호출한 본인이 아닌 다른 CTU가 작업 중인 통로(X좌표) 목록을 수집
        occupied_aisles = set()
        for c in self.ctus:
            if c.id != ctu.id and c.busy: # 자신이 아니며, 현재 작업을 수행 중인 CTU인 경우
                occupied_aisles.update(c.current_batch_aisles) # 해당 CTU가 점유 중인 통로 목록을 블랙리스트(occupied_aisles)에 병합

        # 전체 대기 큐의 작업들 중에서 점유(블랙리스트)되지 않은 통로에 위치한 작업들만 골라 가용 풀을 생성
        available_jobs = [j for j in self.job_queue if j['pos'][0] not in occupied_aisles]

        # 만약 겹치지 않는 작업이 단 하나도 없다면 당장 할당을 포기하고 빈 리스트를 반환하여 안전하게 대기 유도
        if not available_jobs:
            return [] 

        # 1. Nearest
        if rule == 'nearest': # 디스패칭 룰이 가장 가까운 작업 우선(Nearest)인 경우
            # [수정사항: 1번 이슈 해결] 소요 시간(Time) 기준으로 가장 빨리 도착할 수 있는 작업 정렬
            available_jobs.sort(key=lambda j: get_estimated_travel_time(ctu.pos, j['pos'])) 
            batch = [available_jobs.pop(0)] # 정렬된 가용 풀에서 제일 앞(가장 가까운) 작업을 Seed(기준 작업)로 뽑아 배치에 첫 번째로 담음
            
            if available_jobs: # 가용 풀에 남은 작업이 더 있다면
                # [수정사항: 1번 이슈 해결]
                available_jobs.sort(key=lambda j: get_estimated_travel_time(batch[0]['pos'], j['pos'])) 
                extra_count = min(len(available_jobs), TOTE_CAPACITY - 1) # 남은 가용 작업 수와 로봇 여유 적재량(7-1=6) 중 작은 값을 추가 작업 수로 확정
                batch.extend(available_jobs[:extra_count]) # Seed 주변의 추가 가용 작업들을 배치 리스트에 병합
                
            # 원본 job_queue에서 최종 할당된 배치의 작업들만 찾아내어 안전하게 삭제
            for j in batch:
                self.job_queue.remove(j)
                
            self.dispatch_log.append({ # 배치 구성 완료 후 배치 로그 기록
                'time': ctu.env.now, 'ctu_id': ctu.id, 'rule': 'nearest', # 시간, 로봇, 적용 룰 기록
                'seed_item': batch[0]['item_id'], 'batch_items': [j['item_id'] for j in batch], # 기준 부품과 전체 배치 부품 목록 기록
                'batch_size': len(batch), # 이 배치의 총 작업 수 기록
            })
            return batch # 완성된 묶음(배치) 반환

        # 2. Density
        elif rule == 'density': # 디스패칭 룰이 작업 밀집도 우선(Density)인 경우
            K = min(10, len(available_jobs)) # 가용 작업 중 로봇에서 가장 가까운 상위 10개만 Seed 후보로 추림
            # [수정사항: 1번 이슈 해결]
            candidate_seeds = sorted(available_jobs, key=lambda j: get_estimated_travel_time(ctu.pos, j['pos']))[:K] 
            
            best_seed = None # 최적의 Seed 작업을 저장할 변수 초기화
            max_density = -1 # 가장 높은 밀집도 점수를 저장할 변수 초기화
            r_y = 2.65 # 같은 통로 내에서 밀집도를 계산할 검색 반경(2.65m) 설정
            
            for seed in candidate_seeds: # 후보 Seed 10개 각각에 대해 주변 작업 밀집도 조사
                density = 0 # 현재 평가 중인 Seed의 밀집도 카운터 0으로 시작
                for job in available_jobs: # 밀집도를 계산할 때도 다른 로봇이 점유한 통로의 작업은 철저히 배제
                    if job['pos'][0] == seed['pos'][0]: # 평가 대상 작업이 Seed 작업과 같은 통로(X좌표)에 있는지 확인
                        dy = abs(job['pos'][1] - seed['pos'][1]) # 같은 통로라면 Y좌표(깊이) 차이 계산
                        if dy <= r_y: # Y축 거리가 반경(2.65m) 이내라면
                            density += 1 # 주변 작업이므로 밀집도 카운트 1 증가
                
                if density > max_density: # 현재 계산한 밀집도가 역대 최대 밀집도보다 높다면
                    max_density = density # 최대 밀집도 갱신
                    best_seed = seed # 최적 Seed를 현재 Seed로 변경
                    
            # [수정사항: 1번 이슈 해결]
            available_jobs.sort(key=lambda j: get_estimated_travel_time(best_seed['pos'], j['pos'])) 
            
            batch_size = min(len(available_jobs), TOTE_CAPACITY) # 가용 남은 물량과 로봇 최대 용량 중 묶을 크기 결정
            best_batch = available_jobs[:batch_size] # 정렬된 가용 큐에서 위에서 결정한 개수만큼 묶음 생성
            
            # 원본 전체 job_queue에서 배치로 뽑혀나간 작업들을 삭제
            for j in best_batch:
                self.job_queue.remove(j)
            
            self.dispatch_log.append({ # 밀집도 기반 배치 생성 로그 기록
                'time': ctu.env.now, 'ctu_id': ctu.id, 'rule': 'density', # 시간, 로봇 ID, 룰 종류 기록
                'seed_item': best_seed['item_id'], 'batch_items': [j['item_id'] for j in best_batch], # 기준 부품 및 배치 내 부품 리스트 기록
                'batch_size': len(best_batch), 'local_density': max_density, # 배치 크기 및 해당 지역의 계산된 밀집도 점수 기록
            })
            return best_batch # 생성된 밀집 배치 반환

        # 3. Load-balance
        elif rule == 'load_balance': # 디스패칭 룰이 최단 소요 시간 예측(Load-balance)인 경우
            K = min(10, len(available_jobs)) # 가용 풀 작업 중 가까운 상위 10개를 Seed 후보로 추출
            # [수정사항: 1번 이슈 해결]
            nearest_jobs = sorted(available_jobs, key=lambda j: get_estimated_travel_time(ctu.pos, j['pos'])) 
            candidate_seeds = nearest_jobs[:K] 
            
            best_batch = [] # 최단 시간으로 처리될 최적의 배치를 저장할 리스트 초기화
            min_eta = float('inf') # 최소 예상 소요 시간(ETA)을 무한대로 초기화 (비교 갱신용)
            best_seed_item = None # 최적 배치에 사용된 Seed 부품 코드를 저장할 변수 초기화
            
            for seed in candidate_seeds: # 10개의 각 후보 Seed를 가상으로 배치로 묶어봄
                # 임시 큐를 가용 작업(available_jobs)만을 대상으로 Seed 기준 가까운 순 정렬
                # [수정사항: 1번 이슈 해결]
                temp_queue = sorted(available_jobs, key=lambda j: get_estimated_travel_time(seed['pos'], j['pos'])) 
                temp_batch = temp_queue[:TOTE_CAPACITY] # 임시 가용 큐에서 최대 용량만큼 가상 묶음 구성
                
                eta = 0 # 이 가상 배치를 처리할 때의 예상 소요 시간(ETA) 0으로 초기화
                current_pos = ctu.pos  # 로봇의 이동을 시뮬레이션하기 위해 가상 위치를 현재 로봇 위치로 설정
                
                for job in temp_batch: # 가상 배치의 작업들을 순서대로 가상 방문
                    x1, y1, z1 = current_pos # 가상 현재 좌표 언패킹
                    x2, y2, z2 = job['pos'] # 가상 이동 목표 좌표 언패킹
                    
                    if x1 == x2: # 통로가 같은지 확인
                        xy_dist = abs(y1 - y2) # 평면 직진 거리 계산
                    else: # 통로가 다르면
                        xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y) # 교차 통로 우회 거리 계산
                        
                    dz = abs(z1 - z2) # 리프트 이동 거리 계산
                    time_to_job = max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z) # 현재 지점에서 목표 작업까지의 시간 계산
                    eta += time_to_job # 계산된 구간 시간을 가상 총 소요 시간(ETA)에 누적
                    current_pos = job['pos']  # 가상 로봇 위치를 방금 계산한 작업 지점으로 갱신
                
                # --- [수정사항: 4번 이슈 해결] 스테이션 복귀 시간 ETA에 반영 ---
                last_x, last_y, last_z = current_pos
                target_station = STATIONS['S01'] if last_x <= 6.0 else STATIONS['S02']
                
                # 복귀 경로 이동 거리 산출 (is_return=True 방식 적용)
                dist_to_cross_aisle = abs(last_y - CROSS_AISLE_Y)
                euclidean_dist_to_station = math.sqrt((last_x - target_station[0])**2 + (CROSS_AISLE_Y - target_station[1])**2)
                return_xy_dist = dist_to_cross_aisle + euclidean_dist_to_station
                return_dz = abs(last_z - target_station[2])
                
                return_time = max(return_xy_dist / CTU_SPEED_XY, return_dz / CTU_SPEED_Z)
                eta += return_time # 총 소요 시간에 복귀 시간을 합산해야 진짜 Makespan 산출이 가능
                # -----------------------------------------------------------------
                
                if eta < min_eta: # 이 가상 배치의 총 예상 시간이 이전 최소값보다 더 짧다면
                    min_eta = eta # 최소 예상 시간 갱신
                    best_batch = temp_batch # 최적의 배치를 방금 만든 가상 배치로 교체
                    best_seed_item = seed['item_id'] # 그때 사용된 Seed 부품명 저장
            
            # 최종 확정된 배치의 부품들을 진짜 대기 큐(job_queue)에서 안전하게 지우기
            for job in best_batch: 
                self.job_queue.remove(job) 
                
            self.dispatch_log.append({ # 로드 밸런스 기반 배치 생성 로그 기록
                'time': ctu.env.now, 'ctu_id': ctu.id, 'rule': 'load_balance', # 기본 메타데이터 기록
                'seed_item': best_seed_item, 'batch_items': [j['item_id'] for j in best_batch], # Seed 정보와 할당 부품 리스트 기록
                'batch_size': len(best_batch), 'eta': min_eta, # 배치 크기와 시뮬레이션 전 예측했던 예상 종료 시간(ETA) 기록
            })
            return best_batch # 가장 빨리 끝나는 최적 배치 반환

#슬로팅 로직 수정 
def generate_fixed_inventory(rule_type, part_list, weekly_prod_plan, bom_df, outbound_plan, seed=1234):
    """
    교수님 피드백 반영: 
    1. 제품별 생산 계획(물량)을 고려
    2. kitting 시 같은 제품의 부품들을 물리적으로 인접하게 배치
    """
    random.seed(seed)
    inventory = {}
    
    # 1. 제품별 주간 생산 물량 계산
    prod_vol = weekly_prod_plan.groupby('제품코드').size().to_dict()
    # 물량이 많은 순서대로 제품 정렬
    sorted_products = sorted(prod_vol.keys(), key=lambda x: prod_vol.get(x, 0), reverse=True)

    # 2. 가용 슬롯 생성 및 정렬 (스테이션에서 가까운 순)
    y_positions = np.linspace(RACK_START_Y, RACK_END_Y, num=24)
    candidate_slots = []
    for x in AISLE_X:
        for y in y_positions:
            for z in SHELF_HEIGHTS:
                candidate_slots.append((x, round(y,2), z))

    def est_time_to_station(pos):
        x, y, z = pos
        min_time = float('inf')
        for st in STATIONS.values():
            dist = abs(y - CROSS_AISLE_Y) + math.sqrt((x - st[0])**2 + (CROSS_AISLE_Y - st[1])**2)
            t = max(dist / CTU_SPEED_XY, abs(z - st[2]) / CTU_SPEED_Z)
            if t < min_time: min_time = t
        return min_time

    candidate_slots.sort(key=est_time_to_station)

    # 3. 전략별 부품 배치 순서 결정 (Parts Ordering)
    parts_order = []
    used_parts = set()

    # generate_fixed_inventory 함수 내부의 Kitting 전략 부분 수정 제안
   # --- generate_fixed_inventory 함수 내부의 kitting 조건절 수정 ---
    if rule_type == "kitting":
        # 1. 함께 출고되는 부품 쌍(Pair)의 빈도 계산
        pair_counter = Counter()
        for _, group in outbound_plan.groupby('No.'):
            parts = sorted(group['부품코드'].unique())
            if len(parts) > 1:
                for pair in combinations(parts, 2):
                    pair_counter[pair] += 1
        
        # 2. 빈도가 높은 쌍부터 확인하며 '묶음(Cluster)' 생성
        # 조건: 한 묶음은 최소 2개(쌍으로 시작), 최대 8개까지
        clusters = []
        assigned_parts = set()
        
        # 빈도순으로 정렬된 부품 쌍들
        sorted_pairs = [pair for pair, count in pair_counter.most_common()]
        
        for p1, p2 in sorted_pairs:
            # 케이스 1: 둘 다 아직 어느 묶음에도 속하지 않음 -> 새로운 묶음 생성 (최소 2개 만족)
            if p1 not in assigned_parts and p2 not in assigned_parts:
                clusters.append([p1, p2])
                assigned_parts.add(p1)
                assigned_parts.add(p2)
            
            # 케이스 2: p1은 이미 묶음에 있고 p2는 없음 -> p1이 있는 묶음에 p2 추가 (최대 8개 체크)
            elif p1 in assigned_parts and p2 not in assigned_parts:
                for cluster in clusters:
                    if p1 in cluster and len(cluster) < 8:
                        cluster.append(p2)
                        assigned_parts.add(p2)
                        break
            
            # 케이스 3: p2는 이미 묶음에 있고 p1은 없음 -> p2가 있는 묶음에 p1 추가 (최대 8개 체크)
            elif p2 in assigned_parts and p1 not in assigned_parts:
                for cluster in clusters:
                    if p2 in cluster and len(cluster) < 8:
                        cluster.append(p1)
                        assigned_parts.add(p1)
                        break
        
        # 3. 생성된 묶음들을 순서대로 parts_order에 추가
        # (묶음 내의 부품들은 parts_order에서 나란히 배치되므로, 결과적으로 창고에서도 가깝게 배치됨)
        parts_order = []
        for cluster in clusters:
            parts_order.extend(cluster)
            
        # 4. 묶이지 않은 나머지 부품들(단독 출고 등)을 뒤에 추가
        remaining_parts = [p for p in part_list if p not in assigned_parts]
        parts_order.extend(remaining_parts)

    elif rule_type == "characteristic":
        # 생산 물량이 가장 많은 Top 제품의 부품들을 전진 배치
        if sorted_products:
            top_p = sorted_products[0]
            top_parts = set(bom_df[bom_df['product_code'] == top_p]['part_code'].unique())
            parts_order = [p for p in part_list if p in top_parts] + \
                          [p for p in part_list if p not in top_parts]
        else:
            parts_order = list(part_list)

    elif rule_type == "freq_time":
        # 부품별 총 수요량 계산 (제품 생산량 합산)
        part_demand = {}
        for p_code, vol in prod_vol.items():
            p_parts = bom_df[bom_df['product_code'] == p_code]['part_code'].unique()
            for p in p_parts:
                part_demand[p] = part_demand.get(p, 0) + vol
        parts_order = sorted(part_list, key=lambda x: part_demand.get(x, 0), reverse=True)

    elif rule_type == "abc":
        # freq_time과 유사하게 정렬 후 등급 개념 부여 (여기서는 정렬만 수행)
        part_demand = {}
        for p_code, vol in prod_vol.items():
            p_parts = bom_df[bom_df['product_code'] == p_code]['part_code'].unique()
            for p in p_parts:
                part_demand[p] = part_demand.get(p, 0) + vol
        parts_order = sorted(part_list, key=lambda x: part_demand.get(x, 0), reverse=True)

    else: # random
        parts_order = list(part_list)
        random.shuffle(parts_order)

    # 4. 부품과 슬롯 매핑 (1:1)
    for part, slot in zip(parts_order, candidate_slots):
        inventory[part] = list(slot)

    return inventory

# ==========================================
# 하루 단위 시뮬레이션 실행
# ==========================================
def run_simulation(date, inventory, dispatch_rule, date_plan): # 특정 일자의 작업을 시뮬레이션하는 메인 구동 함수
    env = simpy.Environment() # 이산 사건 시뮬레이션의 근간이 되는 SimPy 환경 객체 생성
    warehouse = Warehouse(env, inventory, date_plan) # 위에서 만든 재고와 일일 계획을 바탕으로 창고 객체 인스턴스화
    
    ctus = [CTU(env, i, warehouse, dispatch_rule) for i in range(2)] # 창고 내에서 주어진 디스패칭 룰에 따라 CTU 객체 2대 생성
    warehouse.ctus = ctus # 창고 시스템이 전체 로봇의 위치(점유 통로)를 알 수 있도록 방금 생성한 CTU 리스트를 창고 속성에 주입
    
    def job_gen(env, wh): # 계획에 맞춰 주문 발생을 모사하는 내부 제너레이터 함수
        # 일관된 결과를 위해 각 날짜마다 시드를 일정하게 맞추어 주문 발생 타이밍 통제
        random.seed(hash(date) % 10000) 
        
        for _, group in wh.plan_df.groupby('No.'): # 일일 주문 계획을 주문번호(No.) 별로 묶어 순회
            yield env.timeout(random.uniform(2, 5))  # 각 주문이 시스템에 인입되는 데 걸리는 무작위 시간(2~5초) 간격 부여
            for _, row in group.iterrows(): # 하나의 주문 묶음 내 포함된 부품 리스트 순회
                p_id = row['부품코드'] # 작업해야 할 부품 코드 추출
                if p_id in wh.inventory: # 재고 맵에 이 부품의 위치가 존재하는지 안전 검사
                    wh.job_queue.append({'item_id': p_id, 'pos': wh.inventory[p_id], 'time': env.now}) # 창고 대기 큐에 이 부품의 위치, 주문 발생 시간 기록

    def monitor(env, warehouse, gen_proc, ctus): # 시뮬레이션 종료 시점을 감시하는 내부 함수
        yield gen_proc # 주문 발생 제너레이터(job_gen)가 모든 주문 인입을 끝낼 때까지 대기
        while warehouse.job_queue or any(c.busy for c in ctus): # 주문 생성이 끝난 후에도 창고 큐에 작업이 남아있거나, 아직 작업 중인 로봇이 있다면
            yield env.timeout(1)  # 1초씩 시간을 흐르게 하며 모니터링 반복 루프 지속 (모두 종료되면 루프 탈출)

    gen_proc = env.process(job_gen(env, warehouse)) # 주문 발생 제너레이터를 SimPy 프로세스로 등록
    env.run(until=env.process(monitor(env, warehouse, gen_proc, ctus))) # 모니터 함수(프로세스)가 종료될 때까지 전체 시뮬레이션 엔진 가동
    
    return { # 하루치 시뮬레이션이 끝난 뒤 분석용 데이터 반환
        'Date': date,  # 시뮬레이션 실행 날짜
        'Time': round(env.now, 1),  # 시뮬레이션의 마지막 순간(총 작업 소요 시간) 반환
        'Distance': round(sum(c.total_distance for c in ctus), 1), # 두 로봇의 이동 거리 총합산 반환
        'Move_log': pd.DataFrame(warehouse.move_log), # 쌓인 로봇 이동 이력 로그를 데이터프레임으로 변환하여 반환
        'Dispatch_log': pd.DataFrame(warehouse.dispatch_log), # 쌓인 배치 할당 로그 데이터프레임 변환 반환
    }

# ==========================================
# 개별 알고리즘 조합 통합 시각화 
# ==========================================
def visualize_rule_combination(move_all_df, slot_rule, disp_rule): # 호출 시마다 특정 슬로팅/디스패칭 조합을 시각화하도록 매개변수 추가
    """
    입력받은 슬로팅(slot_rule)과 디스패칭(disp_rule) 조합의 시뮬레이션 결과를 
    3가지 방식(3D Scatter, 2D Trajectory, Gantt Chart)으로 시각화합니다.
    (기존 Kitting + Load_balance 전용 함수를 모든 조합에 범용적으로 쓸 수 있게 확장)
    """
    # 1. 데이터 필터링: 입력받은 조합(slot_rule, disp_rule) 추출
    target_df = move_all_df[(move_all_df['Slotting'] == slot_rule) &  # 전체 이력에서 슬로팅 룰 조건 필터링
                            (move_all_df['Dispatching'] == disp_rule)].copy() # 디스패칭 룰 조건까지 AND 결합하여 복사
    
    if target_df.empty: # 해당 조합의 데이터가 비어있다면
        print(f"해당 조건({slot_rule} + {disp_rule})의 데이터가 없습니다.") # 에러 메시지 출력 후 스킵
        return # 함수 종료
        
    # 시각화가 너무 복잡해지는 것을 막기 위해 첫 번째 시뮬레이션 날짜만 선택
    sample_date = target_df['Date'].iloc[0] # 타겟 데이터프레임에서 가장 첫 번째 줄의 시뮬레이션 날짜 문자열 추출
    df_day = target_df[target_df['Date'] == sample_date].copy() # 전체 데이터 중 방금 추출한 단일 날짜의 결과만 필터링하여 복사
    
    print(f"\n=== [시각화 진행 중: Date {sample_date} | {slot_rule} × {disp_rule}] ===") # 현재 시각화하는 조합과 날짜 정보를 터미널에 명확히 안내

    # ---------------------------------------------------------
    # [1] 3D Scatter Plot: 부품 피킹 위치 분포
    # ---------------------------------------------------------
    fig1 = plt.figure(figsize=(10, 8)) # 3D 산점도를 그리기 위해 가로 10, 세로 8 크기의 새 피규어 생성
    ax1 = fig1.add_subplot(111, projection='3d') # 위 피규어의 첫 번째 영역에 3D 투영(projection)을 활성화한 축 객체 생성
    picks = df_day[df_day['item_id'] != 'RETURN'] # 스테이션 복귀는 제외하고 실제 부품 좌표 데이터만 추출
    pick_counts = picks.groupby(['to_x', 'to_y', 'to_z', 'item_id']).size().reset_index(name='count') # 동일한 X,Y,Z 위치에 피킹이 몇 번 일어났는지 집계하여 count 열로 명명
    
    scatter = ax1.scatter(pick_counts['to_x'], pick_counts['to_y'], pick_counts['to_z'],  # 추출한 x, y, z 좌표 리스트를 기반으로 3D 산점도 그리기
                          s=pick_counts['count']*50, c=pick_counts['to_z'], cmap='viridis', alpha=0.7) # 빈도(count)가 클수록 점 크기(s) 확대, 층수(Z)에 따라 색상 분리, 반투명도 지정
    
    ax1.set_title(f'① 3D Scatter: 피킹 위치 분포 ({slot_rule} × {disp_rule})', fontsize=14, fontweight='bold') # 3D 플롯 제목에 현재 적용 중인 룰 변수를 동적 삽입
    ax1.set_xlabel('통로 X (Aisle)') # 3D 플롯 X축 레이블 지정
    ax1.set_ylabel('랙 깊이 Y') # 3D 플롯 Y축 레이블 지정
    ax1.set_zlabel('선반 높이 Z') # 3D 플롯 높이 Z축 레이블 지정
    fig1.colorbar(scatter, ax=ax1, label='Z축 높이 (층수)') # Z 높이에 따른 범례 역할을 하는 컬러바를 우측에 부착
    plt.tight_layout() # 요소 간 겹침 방지 여백 조정
    plt.show() # 3D 산점도 화면 출력

    # ---------------------------------------------------------
    # [2] 2D Trajectory Map: 로봇 이동 궤적 밀집도 (Heatmap 스타일 렌더링)
    # ---------------------------------------------------------
    fig2, ax2 = plt.subplots(figsize=(12, 8)) # 로봇 궤적을 그릴 12x8 사이즈 피규어와 2D 축 생성
    
    # 랙 및 스테이션 배경 그리기: 기획안 도면(R01 ~ R12)과 완전히 동일한 X좌표와 폭(Width)으로 백투백(Back-to-Back) 랙 덩어리들을 생성
    rack_spans = [(0, 0.6), (1.8, 1.2), (4.2, 1.2), (6.6, 1.2), (9.0, 1.2), (11.4, 1.2), (13.8, 0.6)] # 구조: (단일 0.6m) - (양면 1.2m) * 5 - (단일 0.6m), 통로 간격은 1.2m씩
    for start_x, width in rack_spans: # 랙 덩어리 좌표 배열을 순회
        ax2.add_patch(patches.Rectangle((start_x, RACK_START_Y), width, RACK_END_Y - RACK_START_Y, color='lightgray')) # 도면에 맞게 회색 랙 덩어리 배치
    
    for x in AISLE_X: # 통로 라벨 추가
        ax2.text(x, RACK_END_Y + 0.5, f"Aisle {x}", ha='center', fontsize=9) # 각 통로 위에 번호 텍스트 부착
        
    ax2.axhline(CROSS_AISLE_Y, color='black', linestyle='--', alpha=0.3, label='교차 통로 (Cross Aisle)') # 로봇이 교차하여 지나가는 Y축(2.0) 위치에 검은 점선 가이드라인 표시
    
    ax2.add_patch(patches.Rectangle((2.6, 0), 2.0, 1.0, color='lightgreen', label='Station (S01, S02)')) # 작업 스테이션 S01 위치에 연녹색 박스 도형 생성 및 배치
    ax2.add_patch(patches.Rectangle((9.8, 0), 2.0, 1.0, color='lightgreen')) # 작업 스테이션 S02 위치에 도형 배치
    ax2.text(3.6, 0.5, 'S01', ha='center', va='center', fontweight='bold') # S01 박스 정중앙에 'S01' 텍스트 레이블 추가
    ax2.text(10.8, 0.5, 'S02', ha='center', va='center', fontweight='bold') # S02 박스 내부 텍스트 레이블 추가

    # ----- 핵심 로직: 궤적을 5cm 단위의 점(Point)으로 쪼개 통행량 카운트 -----
    traffic_counts = {} # 특정 (x, y) 좌표를 지날 때마다 누적 횟수를 저장할 빈 딕셔너리 생성

    def add_line_segment(sx, sy, ex, ey): # 시뮬레이션의 선형 궤적을 점 단위로 분해하는 내부 함수
        """시작점과 끝점을 받아 그 사이의 경로를 5cm 간격의 점으로 변환 후 카운트 증가"""
        # [수정사항] 궤적 렌더링 시에도 대각선 이동을 매끄럽게 그릴 수 있도록 맨해튼 방식이 아닌 유클리드 거리로 점의 개수를 산출
        dist = math.sqrt((sx - ex)**2 + (sy - ey)**2) 
        steps = int(dist / 0.05) + 1 # 0.05m(5cm) 해상도로 경로를 쪼개어 생성할 점의 전체 개수 계산
        for i in range(steps): # 계산된 점의 개수만큼 루프 반복
            px = sx + (ex - sx) * i / max(1, steps - 1) # i번째 점의 X 좌표 보간 계산
            py = sy + (ey - sy) * i / max(1, steps - 1) # i번째 점의 Y 좌표 보간 계산
            coord = (round(px, 2), round(py, 2)) # 계산된 점을 소수점 2자리로 정규화하여 튜플 생성
            traffic_counts[coord] = traffic_counts.get(coord, 0) + 1 # 딕셔너리에 해당 좌표 키가 있으면 값을 1 증가, 없으면 1로 초기화 (통행 빈도 누적)

    # 로봇의 모든 점대점 이동 이력 순회
    for _, row in df_day.iterrows(): # 해당 날짜 로봇의 모든 이동 이력을 한 줄씩 순회
        x1, y1 = row['from_x'], row['from_y'] # 출발점 X, Y 좌표 추출
        x2, y2 = row['to_x'], row['to_y'] # 도착점 X, Y 좌표 추출
        
        if row['item_id'] == 'RETURN': 
            # [수정사항] 복귀 궤적: 랙에서 빠져나와 교차통로(Y=3.0)까지는 직진 이동 후, 스테이션까지 유클리드(대각선) 궤적을 추가
            add_line_segment(x1, y1, x1, CROSS_AISLE_Y) # 랙에서 교차 통로로 나오기
            add_line_segment(x1, CROSS_AISLE_Y, x2, y2) # 교차 통로에서 스테이션으로 대각선 횡단
        else:
            if x1 == x2: # 출발 통로와 도착 통로가 같아 꺾을 필요가 없다면 (한 번에 직진)
                add_line_segment(x1, y1, x2, y2) # 해당 직진 구간 전체를 점으로 쪼개 카운트 등록
            else: # 다른 통로로 이동하는 경우 (교차통로 우회 3단계 맨해튼 궤적)
                add_line_segment(x1, y1, x1, CROSS_AISLE_Y)        # 1. 현재 랙 안쪽에서 교차로 지점(Y=3.0)까지 후퇴하는 직선 구간 카운트
                add_line_segment(x1, CROSS_AISLE_Y, x2, CROSS_AISLE_Y) # 2. 교차로 지점 내에서 목표 통로(X2)까지 가로로 횡단하는 구간 카운트
                add_line_segment(x2, CROSS_AISLE_Y, x2, y2)        # 3. 목표 통로 교차로에서 최종 랙 안쪽(Y2)으로 깊숙이 진입하는 직선 구간 카운트

    # 수집된 통행 빈도 데이터를 시각화를 위한 리스트로 분리
    pts_x, pts_y, pts_c = [], [], [] # X좌표 리스트, Y좌표 리스트, 빈도수(색상값) 리스트 초기화
    for (x, y), count in traffic_counts.items(): # 누적된 통행 딕셔너리를 순회
        pts_x.append(x) # 분리된 X 리스트에 추가
        pts_y.append(y) # 분리된 Y 리스트에 추가
        pts_c.append(count) # 빈도수를 리스트에 추가

    # -------------------------------------------------------------------
    # Scatter Plot을 활용해 선 대신 연속된 점들로 그라데이션(Heatmap) 렌더링
    # cmap='Reds': 한 번 지나가면 연한 핑크, 여러 번 지나가 병목인 곳은 진한 빨강
    # -------------------------------------------------------------------
    sc = ax2.scatter(pts_x, pts_y, c=pts_c, cmap='Reds', s=15, zorder=3, alpha=0.9) # 추출한 리스트를 산점도로 그리되 빈도(pts_c)에 따라 빨간색 농도를 다르게 적용
    
    # 우측에 컬러바를 부착하여 어떤 색상이 얼마나 많은 횟수를 의미하는지 범례 제공
    cbar = fig2.colorbar(sc, ax=ax2) # 방금 그린 산점도 색상 기준을 축으로 삼아 컬러바 생성
    cbar.set_label('로봇 누적 통행 빈도 (Traffic Intensity)') # 컬러바 옆 설명 텍스트 부착

    ax2.set_title(f'② 2D Trajectory Map: 로봇 통행 밀집도 궤적 ({slot_rule} × {disp_rule})', fontsize=14, fontweight='bold') # 2D 궤적 맵 제목에 현재 적용 중인 조합 변수명 동적 표시
    ax2.set_xlabel('X 좌표 (m)') # 궤적 맵 X축 물리적 단위(미터) 라벨
    ax2.set_ylabel('Y 좌표 (m)') # 궤적 맵 Y축 라벨
    ax2.set_xlim(-1, 15.5) # 맨 우측 랙이 잘리지 않도록 X축 오른쪽 여백 확대
    ax2.set_ylim(-1, 17) # 하단 스테이션 영역과 랙 상단 여백 표시를 위해 Y 범위 -1 ~ 17 확보
    
    # 기존 범례를 깔끔하게 유지하기 위한 정리 로직
    handles, labels = ax2.get_legend_handles_labels() # 플롯에 추가된 모든 막대 종류의 범례 핸들과 라벨 텍스트 수집
    by_label = dict(zip(labels, handles)) # 중복 라벨을 하나로 합치기 위해 딕셔너리 구조 활용
    ax2.legend(by_label.values(), by_label.keys(), loc='upper left') # 좌측 상단에 정돈된 범례 표시
    
    plt.tight_layout() # 간격 자동 조절
    plt.show() # 2D 궤적 맵 화면 출력

    # ---------------------------------------------------------
    # [3] Gantt Chart: CTU별 작업 시간 및 유휴 시간 확인
    # ---------------------------------------------------------
    fig3, ax3 = plt.subplots(figsize=(12, 4)) # 가로로 긴 형태(12x4)의 간트 차트 피규어 생성
    ctu_tasks = {0: [], 1: []} # 각 로봇의 (작업 시작 시점, 작업 소요 시간) 정보 튜플을 담을 빈 리스트 딕셔너리
    ctu_returns = {0: [], 1: []} # 복귀 구간도 별도의 색상으로 구분하기 위해 (시작, 소요) 튜플을 담을 딕셔너리
    
    for _, row in df_day.iterrows(): # 모든 이동 이력을 시간순으로 순회
        end_time = row['time'] # 해당 구간 이동이 끝난 완료 시간
        duration = row['travel_time'] # 해당 구간 이동에 걸린 소요 시간(초)
        start_time = end_time - duration # 이동 완료 시점에서 소요 시간을 빼 이동이 시작된 출발 시간 역산
        
        if row['item_id'] == 'RETURN': # 만약 이동 목적이 작업이 아닌 스테이션 복귀라면
            ctu_returns[row['ctu_id']].append((start_time, duration)) # 복귀 리스트에 시작, 지속시간 튜플 저장
        else: # 일반 부품 피킹 이동이라면
            ctu_tasks[row['ctu_id']].append((start_time, duration)) # 작업 리스트에 시작, 지속시간 저장
            
    ax3.broken_barh(ctu_tasks[0], (5, 8), facecolors='tab:blue', label='CTU 0 (피킹)') # CTU 0번 작업 리스트의 막대들을 Y축 기준 5~13 위치에 파란색으로 그림
    ax3.broken_barh(ctu_returns[0], (5, 8), facecolors='lightblue', label='CTU 0 (복귀)') # CTU 0번 복귀 구간 막대를 같은 Y축 라인에 연파랑으로 겹쳐서 표시
    ax3.broken_barh(ctu_tasks[1], (15, 8), facecolors='tab:red', label='CTU 1 (피킹)') # CTU 1번 작업 막대를 Y축 15~23 윗줄에 빨강으로 배치
    ax3.broken_barh(ctu_returns[1], (15, 8), facecolors='lightcoral', label='CTU 1 (복귀)') # CTU 1번 복귀 구간 연빨강 처리
    
    ax3.set_ylim(0, 30) # Y축의 범위를 0부터 30까지 넉넉하게 설정하여 두 줄의 막대가 중앙에 잘 보이도록 여백 확보
    ax3.set_xlim(0, df_day['time'].max() + 10) # X축(시간)을 0초부터 가장 마지막 기록된 종료 시간의 +10초 여유분까지 설정
    ax3.set_yticks([9, 19]) # Y축 좌측 라벨이 막대의 정중앙(5+4, 15+4)을 가리키도록 틱 위치 9, 19 지정
    ax3.set_yticklabels(['CTU 0', 'CTU 1']) # 지정된 틱 위치에 로봇 이름 텍스트 표시
    ax3.set_xlabel('시뮬레이션 시간 (초)') # X축이 시간 흐름임을 명시
    ax3.set_title(f'③ Gantt Chart: CTU 작업 흐름 ({slot_rule} × {disp_rule})', fontsize=14, fontweight='bold') # 간트 차트 제목에 조합 변수명 동적 표시
    
    handles, labels = ax3.get_legend_handles_labels() # 플롯에 추가된 모든 막대 종류의 범례 핸들과 라벨 텍스트 수집
    by_label = dict(zip(labels, handles)) # 중복 라벨을 하나로 합치기 위해 딕셔너리 구조 활용 (같은 이름이면 덮어씀)
    ax3.legend(by_label.values(), by_label.keys(), loc='upper right') # 중복 제거된 유니크한 범례를 우측 상단에 박스로 표시
    
    plt.tight_layout() # 차트 여백 최적화
    plt.show() # 최종 간트 차트 팝업 출력

# ==========================================
# 로그 분석 함수
# ==========================================
def analyze_logs(move_df, dispatch_df, title_prefix=""): # 분석 텍스트 결과를 터미널에 프린트하는 유틸리티 함수
    print(f"\n{'='*60}") # 출력 구분 상단선
    print(f" [{title_prefix}] 분석 결과") # 분석 테마 제목
    print(f"{'='*60}") # 출력 제목 하단선
    
    if not move_df.empty: # 이동 로그가 정상적으로 있다면
        print(f"\n CTU별 총 이동거리:") # 소제목 출력
        ctu_dist = move_df.groupby('ctu_id')['travel_distance'].sum() # 각 로봇의 이동 거리 총합을 그룹 연산
        print(ctu_dist.round(1)) # 구한 이동 거리를 소수점 1자리로 프린트
        
        print(f"\n 평균 배치당 이동거리:") # 소제목 출력
        avg_dist_per_batch = move_df[move_df['item_id'] != 'RETURN'].groupby( # 복귀가 아닌 실제 작업의 이동 거리를
            ['dispatch_rule', 'ctu_id'] # 디스패칭 방식별, 로봇별로 그룹화하여
        )['travel_distance'].mean() # 평균값 산출
        print(avg_dist_per_batch.round(1)) # 평균 거리 프린트
    
    if not dispatch_df.empty: # 배치 할당 로그가 존재한다면
        print(f"\n 배치 크기 통계:") # 소제목 출력
        batch_stats = dispatch_df.groupby('rule')['batch_size'].agg(['mean', 'std', 'min', 'max']).round(1) # 디스패칭 룰에 따라 한 번에 몇 개를 묶어갔는지 평균, 표준편차, 최소/최대값 집계
        print(batch_stats) # 통계 테이블 프린트

# ==========================================
# 재배치 주기별 시각화 

def visualize_combo_3d_comparison(move_all_df, s_rule, d_rule):
    """
    31일 전체 데이터를 1, 2, 3, 4주차로 나누어 
    주차별 피킹 빈도 변화를 4분할 화면으로 시각화합니다.
    """
    # 1. 해당 조합 데이터 필터링 (RETURN 제외)
    df_combo = move_all_df[(move_all_df['Slotting'] == s_rule) & 
                           (move_all_df['Dispatching'] == d_rule) &
                           (move_all_df['item_id'] != 'RETURN')].copy()
    
    if df_combo.empty: return

    # 2. 주차별 데이터 슬라이싱 (7일 단위)
    unique_dates = sorted(df_combo['Date'].unique())
    
    # 각 주차별 날짜 리스트 정의
    week_list = [
        unique_dates[0:7],   # 1주차 (1~7일)
        unique_dates[7:14],  # 2주차 (8~14일)
        unique_dates[14:21], # 3주차 (15~21일)
        unique_dates[21:]    # 4주차 및 잔여일 (22~31일)
    ]
    
    titles = ["1주차 (Day 1~7)", "2주차 (Day 8~14)", "3주차 (Day 15~21)", "4주차+ (Day 22~31)"]

    # 4개의 그래프를 보여주기 위해 2x2 레이아웃으로 설정
    fig = plt.figure(figsize=(22, 16))
    
    for i, (week_dates, title) in enumerate(zip(week_list, titles)):
        # 2행 2열의 i+1번째 자리에 서브플롯 생성
        ax = fig.add_subplot(2, 2, i+1, projection='3d')
        
        df_week = df_combo[df_combo['Date'].isin(week_dates)]
        if df_week.empty: continue
        
        # 좌표별 피킹 횟수 집계
        pick_counts = df_week.groupby(['to_x', 'to_y', 'to_z', 'item_id']).size().reset_index(name='freq')
        
        # 3D 산점도 그리기
        sc = ax.scatter(pick_counts['to_x'], pick_counts['to_y'], pick_counts['to_z'], 
                        c=pick_counts['freq'], 
                        cmap='YlOrRd', 
                        s=pick_counts['freq'] * 20 + 15, # 점 크기 조절
                        alpha=0.7, edgecolors='grey', linewidth=0.2)
        
        # 빈도 상위 3개 위치에만 라벨 표시 (화면이 좁아지므로 숫자를 줄임)
        top_3 = pick_counts.nlargest(3, 'freq')
        for _, row in top_3.iterrows():
            ax.text(row['to_x'], row['to_y'], row['to_z'] + 0.3, f"{row['item_id']}({int(row['freq'])})", 
        fontsize=8, fontweight='bold')

        ax.set_title(f"{title}\n[{s_rule} × {d_rule}]", fontsize=12, fontweight='bold')
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        ax.view_init(elev=20, azim=-45)

    plt.suptitle(f"알고리즘 조합별 주차별(1~4W) 피킹 분포 추이 비교", fontsize=20, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    
def report_kpi_scoreboard(res_df):
    """31일간의 알고리즘 성과를 수치로 요약 (KPI) - 1등 색상 강조 버전"""
    import os
    os.system('') # 윈도우 터미널 색상 활성화
    
    print("\n" + "="*85)
    print(" [최종 분석 보고: 31일간의 알고리즘 성과 스코어보드] ")
    print("="*85)
    
    # 1. 데이터 계산
    kpi = res_df.groupby(['Slotting', 'Dispatching'])[['Distance', 'Time']].mean().round(1)
    base_dist = kpi.loc[('random', 'nearest'), 'Distance']
    kpi['성능개선율(%)'] = ((base_dist - kpi['Distance']) / base_dist * 100).round(1)
    kpi = kpi.sort_values(by='Distance')

    # 색상 코드 정의
    YELLOW = '\033[93m' # 밝은 노란색 글자
    RESET = '\033[0m'   # 색상 초기화

    # 2. 결과 출력 (1등만 색상 적용)
    # 헤더 출력
    print(f"{'Slotting':<15} {'Dispatching':<15} {'Distance':>12} {'Time':>12} {'성능개선율(%)':>15}")
    print("-" * 85)

    for i, ((slot, disp), row) in enumerate(kpi.iterrows()):
        # 한 줄 문자열 생성
        line = f"{slot:<15} {disp:<15} {row['Distance']:>12.1f} {row['Time']:>12.1f} {row['성능개선율(%)']:>14}%"
        
        if i == 0: # 1등일 때만 노란색으로 출력
            print(f"{YELLOW}{line} ★ 최적{RESET}")
        else:
            print(line)
    
    print("-" * 85)
    best = kpi.index[0]
    print(f"\n★ 최적 조합: {YELLOW}[{best[0]} x {best[1]}]{RESET}")
    print(f"   => 베이스라인 대비 {YELLOW}{kpi.loc[best, '성능개선율(%)']}%{RESET} 효율 향상 증명")


# ==========================================
# main 함수 (수정)
# ==========================================
def main():
    # 1. 데이터 파일 읽기
    outbound_plan = pd.read_csv('outbound_plan.csv')
    production_plan = pd.read_csv('production_plan.csv')
    bom_df = pd.read_excel('BOM.xlsx')

    outbound_plan['생산일자'] = pd.to_datetime(outbound_plan['생산일자']).dt.date
    production_plan['생산일자'] = pd.to_datetime(production_plan['생산일자']).dt.date
    unique_dates = sorted(outbound_plan['생산일자'].unique())
    part_list = outbound_plan['부품코드'].unique().tolist() 
    
    # 3. 실험할 알고리즘 조합 정의
    slotting_rules = ['random', 'abc', 'characteristic', 'kitting', 'freq_time']
    dispatch_rules = ['nearest', 'density', 'load_balance']

    all_results_summary = []
    move_logs_all = []
    dispatch_logs_all = []

    print(f"총 {len(unique_dates)}일치 데이터에 대해 시뮬레이션을 시작합니다.")

    # 4. 시뮬레이션 실행 루프 (슬로팅 x 디스패칭)
    for s_rule in slotting_rules:
        for d_rule in dispatch_rules:
            inventory = None 
            for i, date in enumerate(unique_dates):
                # [핵심] 7일마다 재배치
                if i % 7 == 0:
                    upcoming_7 = unique_dates[i : i+7]
                    weekly_plan = production_plan[production_plan['생산일자'].isin(upcoming_7)]
                    inventory = generate_fixed_inventory(s_rule, part_list, weekly_plan, bom_df, outbound_plan, seed=100)
                
                    print(f">> {date} [{s_rule} x {d_rule}] 실행 중...")

                date_plan = outbound_plan[outbound_plan['생산일자'] == date]
                res = run_simulation(date, inventory, d_rule, date_plan)

                # 결과 수집
                all_results_summary.append({
                    'Date': res['Date'], 'Time': res['Time'], 'Distance': res['Distance'],
                    'Slotting': s_rule, 'Dispatching': d_rule,
                })

                move_tmp = res['Move_log'].copy()
                move_tmp['Slotting'], move_tmp['Dispatching'], move_tmp['Date'] = s_rule, d_rule, date
                move_logs_all.append(move_tmp)

                disp_tmp = res['Dispatch_log'].copy()
                disp_tmp['Slotting'], disp_tmp['Dispatching'], disp_tmp['Date'] = s_rule, d_rule, date
                dispatch_logs_all.append(disp_tmp)

    # 5. 최종 분석 및 1등 조합 시각화 (중복 없이 한 번만 실행)
    df_res = pd.DataFrame(all_results_summary)
    move_all_df = pd.concat(move_logs_all, ignore_index=True)
    dispatch_all_df = pd.concat(dispatch_logs_all, ignore_index=True)

    # (1) KPI 성적표 출력
    report_kpi_scoreboard(df_res)
    
    # (2) 자동으로 최적 조합 찾기
    best_combo = df_res.groupby(['Slotting', 'Dispatching'])['Distance'].mean().idxmin()
    best_s, best_d = best_combo 

    print(f"\n" + "="*60)
    print(f" 최적 조합 분석 결과: [{best_s} x {best_d}]")
    print(f" 해당 조합에 대한 상세 리포트 그래프를 생성합니다.")
    print("="*60)

    # (3) 1등 조합에 대해서만 시각화 (창 4개만 뜸)
    analyze_logs(move_all_df, dispatch_all_df, title_prefix=f"최적({best_s}x{best_d})")
    visualize_combo_3d_comparison(move_all_df, best_s, best_d) # 주차별 비교
    visualize_rule_combination(move_all_df, best_s, best_d)    # 3D/2D/간트

if __name__ == "__main__":
    main()

기존 문제: 로봇의 주행 속도와 리프트 승강 속도가 다름에도 불구하고, 소요 시간이 아닌 **단순 물리적 거리(맨해튼 거리)**의 합만으로 작업을 정렬하여 비효율적인 선택(거리는 가깝지만 시간은 오래 걸리는 작업)이 발생할 수 있었습니다.

수정 결과: 거리 계산식을 폐기하고 로봇의 실제 속도가 반영된 '예상 소요 시간' 기준으로 공식을 완전히 교체하여, 진짜로 제일 빨리 도달할 수 있는 작업을 정확히 찾도록 로직을 수정했습니다.

load balance룰이 작업 후 스테이션 복귀시간이 반영 안 되어있어서 스테이션 복귀시간 추가해서 작업 고르도록 로직 수정

In [ ]:
import simpy
import pandas as pd 
import numpy as np
import random
import math                         # [수정사항] 유클리드 대각선 거리를 계산하기 위해 math 모듈 추가
from itertools import combinations  # 주문 내에서 함께 출고되는 부품쌍 생성 (Kitting Slotting에 사용)
from collections import Counter     # 함께 출고되는 부품쌍의 출현 빈도 계산 (어떤 부품을 서로 가깝게 배치할지 판단하기 위함)
import matplotlib.pyplot as plt     # 시각화 및 스타일링
import seaborn as sns               # 히트맵용 시각화
import matplotlib.patches as patches # 2D 궤적 맵에서 랙(Rack)을 그리기 위함

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# --- 시스템 환경 설정 (상수 정의) ---
# 창고의 물리적 레이아웃과 물류 로봇 성능 정의
RACK_START_Y, RACK_END_Y = 4.0, 15.1       # 랙이 설치된 Y축의 시작과 끝 좌표 범위 (4.0m ~ 15.1m 사이에만 적재 가능)
AISLE_X = [1.2, 3.6, 6.0, 8.4, 10.8, 13.2] # 창고 통로(Aisle)가 위치한 X축 좌표 리스트

# 작업 스테이션 위치 (x, y, z): 레이아웃과 일치하도록 S01은 A02(3.6)와, S02는 A05(10.8)와 X좌표를 완벽히 맞춤
STATIONS = {'S01': [3.6, 0.5, 0], 'S02': [10.8, 0.5, 0]} 

CTU_SPEED_XY = 1.8  # 로봇의 XY 이동 속도 (m/s)
CTU_SPEED_Z = 0.5   # 로봇의 리프트(Z) 상하 이동 속도 (m/s)
TOTE_CAPACITY = 7   # CTU가 한 번에 운반할 수 있는 최대 토트 박스 수
SHELF_HEIGHTS = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0] # 선반의 층별 높이(Z축) 리스트 (부품의 Z좌표로 사용)

# 교차 통로(Cross Aisle) 설정
# 로봇이 다른 통로로 넘어갈 때 랙을 관통하지 않고 앞쪽으로 빠져나와서 이동하는 기준 Y 좌표 
CROSS_AISLE_Y = 3.0

# ==========================================
# 창고 레이아웃 기반 예상 소요 시간 산출 함수 (기존 거리 함수 대체)
# ==========================================
def get_estimated_travel_time(pos1, pos2): # [수정사항] 단순 거리에서 속도가 반영된 소요 시간 계산으로 변경
    """
    물류 로봇이 랙(Rack)을 뚫고 지나갈 수 없다는 물리적 제약을 반영하여
    두 위치(pos1, pos2) 사이의 '예상 이동 소요 시간'을 계산하는 함수.
    디스패칭 시 어떤 작업이 가장 '빨리' 도달할 수 있는지 판단하는 기준이 됨.
    """
    x1, y1, z1 = pos1 # 시작 지점의 x, y, z 좌표를 언패킹
    x2, y2, z2 = pos2 # 도착 지점의 x, y, z 좌표를 언패킹
    
    # 1. 같은 통로(X좌표 동일)일 경우: 단순히 Y축 깊이 차이만 계산
    if x1 == x2: # 시작점과 도착점의 X 좌표(통로)가 같은지 확인
        xy_dist = abs(y1 - y2) # 같은 통로면 Y 좌표의 차이 절대값을 평면 거리로 산출
    # 2. 다른 통로일 경우: 현재 통로에서 교차 통로(Y=3.0)로 직진하여 빠져나온 뒤, 가로(X)로 이동 후 새 통로로 진입
    else: # 다른 통로로 이동해야 하는 경우
        xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y) # 현재 Y에서 교차통로로 나가는 거리 + X축 이동 거리 + 새 통로로 들어가는 거리 합산
        
    dz = abs(z1 - z2) # Z축(높이) 차이 계산
    
    # 평면 이동 속도와 리프트 속도 중 병목이 되는(더 오래 걸리는) 진짜 소요 시간을 반환
    return max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z) 

# ==========================================
# CTU class 물류 로봇
# ==========================================
class CTU: # 물류 로봇(CTU) 객체를 정의하는 클래스
    def __init__(self, env, id, warehouse, dispatch_rule): # 로봇 초기화 메서드 (환경, ID, 소속 창고, 디스패칭 룰을 인자로 받음)
        self.env = env # SimPy 시뮬레이션 환경 객체를 로봇의 속성으로 저장
        self.id = id # CTU를 구별하기 위한 고유 번호(CTU 0 또는 1)
        self.warehouse = warehouse # 로봇이 작업할 창고 객체의 참조를 저장
        self.dispatch_rule = dispatch_rule # 현재 로봇에 적용할 작업 할당 방식(Nearest, Density 등) 저장
        
        # 초기 로봇 위치 설정: 0번 CTU는 S01에서, 1번 CTU는 S02에서 시작
        self.pos = STATIONS['S01'] if id == 0 else STATIONS['S02'] # ID가 0이면 S01 좌표, 아니면 S02 좌표를 초기 위치로 설정

        # 총 이동거리 및 작업 수 기록
        self.total_distance = 0 # 로봇이 이동한 총 누적 거리를 0으로 초기화
        self.completed_jobs = 0 # 로봇이 완료한 총 피킹 작업 수를 0으로 초기화
        self.busy = False # 로봇의 현재 상태를 대기 중(False)으로 초기화
        
        # 이 CTU가 현재 작업하기 위해 점유 중인 통로(X좌표)들을 담아두는 집합(Set) 변수 추가
        self.current_batch_aisles = set() 
        
        self.process = env.process(self.run()) # 로봇의 메인 작업 루프인 run()을 SimPy 이벤트 큐에 프로세스로 등록

    # 이동 시간 및 거리 계산 (맨해튼 직교 거리 기반)
    def calculate_travel_time(self, target_pos, is_return=False): # 복귀 여부를 알 수 있는 is_return 파라미터 기본값 False로 추가
        x1, y1, z1 = self.pos # 로봇의 현재 위치 좌표 언패킹
        x2, y2, z2 = target_pos # 로봇이 이동해야 할 목표 위치 좌표 언패킹
        
        if is_return: 
            # 스테이션으로 복귀할 경우: 
            # 랙 내부(Y>=4.0)에서 교차 통로(Y=3.0)까지는 통로를 따라 직진(Manhattan)으로 빠져나오고,
            # Y=3.0 교차 통로에서부터 스테이션(Y=0.5) 구간은 장애물이 없는 열린 공간이므로 대각선(유클리드)으로 가로지릅니다.
            dist_to_cross_aisle = abs(y1 - CROSS_AISLE_Y) 
            euclidean_dist_to_station = math.sqrt((x1 - x2)**2 + (CROSS_AISLE_Y - y2)**2)
            xy_dist = dist_to_cross_aisle + euclidean_dist_to_station
        else:
            if x1 == x2: # 출발지와 목적지의 통로(X좌표)가 동일한지 확인
                xy_dist = abs(y1 - y2) # 통로가 같으면 랙 안쪽으로 직진하는 거리만 계산
            else: # 통로가 다른 경우
                xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y) # 앞쪽 교차 통로(Y=3.0)로 우회하는 평면 궤적 거리 계산
            
        dz = abs(z1 - z2) # Z축(상하 리프트) 이동 거리 계산
        
        travel_time = max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z) # 평면 주행과 리프트 승강은 동시 수행되므로 둘 중 더 오래 걸리는 시간을 소요 시간으로 산정
        travel_distance = xy_dist + dz # 물리적인 총 모터 이동 거리(주행 거리 + 리프트 이동 거리) 합산
        
        return travel_time, travel_distance # 계산된 이동 시간과 총 이동 거리를 튜플로 반환

    # CTU의 실제 작업 수행 프로세스
    def run(self): # 시간 흐름에 따라 CTU가 지속적으로 수행할 제너레이터 메서드
        while True: # 시뮬레이션 종료 시까지 무한 반복
            if self.warehouse.job_queue: # 창고에 처리 대기 중인 주문 작업이 남아있는지 확인
                # 무조건 busy를 True로 바꾸지 않고, 겹치지 않는 배치를 받아올 수 일 때만 상태를 변경하도록 로직 순서 변경
                batch = self.warehouse.apply_dispatching(self, self.dispatch_rule) 
                
                if batch: # 겹치지 않는 할당받은 작업 묶음이 존재한다면
                    self.busy = True # 비로소 작업 상태(True)로 전환
                    
                    # 할당받은 작업 배치에 포함된 모든 통로의 X좌표를 추출하여 현재 CTU의 점유 통로로 등록
                    # (이후 다른 CTU가 디스패칭을 수행할 때 이 통로의 작업들은 투명 인간처럼 무시됨)
                    self.current_batch_aisles = set(job['pos'][0] for job in batch)
                    
                    # [수정사항: 1번 이슈 해결] 거리 기준 정렬에서 '예상 소요 시간' 기준 정렬로 변경
                    batch.sort(key=lambda j: get_estimated_travel_time(self.pos, j['pos'])) 
                    
                    for job in batch: # 할당받은 각 작업(부품 피킹)을 순회
                        t, d = self.calculate_travel_time(job['pos']) # 해당 부품 위치까지의 이동 시간(t)과 거리(d)를 계산
                        
                        self.warehouse.move_log.append({ # 분석을 위해 로봇의 이동 상세 이력을 창고의 로그 리스트에 추가
                            'time': self.env.now, # 현재 시뮬레이션 시간 기록
                            'ctu_id': self.id, # 어떤 로봇이 이동했는지 ID 기록
                            'from_x': self.pos[0], 'from_y': self.pos[1], 'from_z': self.pos[2], # 출발 지점 X, Y, Z 좌표 기록
                            'to_x': job['pos'][0], 'to_y': job['pos'][1], 'to_z': job['pos'][2], # 도착(피킹) 지점 X, Y, Z 좌표 기록
                            'item_id': job['item_id'], # 피킹한 대상 부품 코드 기록
                            'travel_time': t, 'travel_distance': d, # 해당 구간 이동 소요 시간과 이동 거리 기록
                            'dispatch_rule': self.dispatch_rule, # 당시 적용된 디스패칭 룰 기록
                        })
                        
                        yield self.env.timeout(t) # 계산된 이동 시간(t)만큼 시뮬레이션 시간을 흘려보냄 
                        
                        self.pos = job['pos'] # 로봇의 현재 위치를 방금 피킹을 완료한 목적지 좌표로 갱신
                        self.total_distance += d # 로봇의 누적 이동 거리에 방금 이동한 거리를 더함
                        self.completed_jobs += 1 # 완료한 피킹 작업 개수 1 증가
                        
                    # 마지막 피킹 위치의 X좌표에 따라 복귀할 지정 스테이션 결정
                    last_job_x = self.pos[0] # 배치의 모든 피킹이 끝난 후 로봇의 현재(마지막) 통로 X 좌표 확인
                    target_station = STATIONS['S01'] if last_job_x <= 6.0 else STATIONS['S02'] # 마지막 X 좌표가 6.0 이하(A01~A03)면 S01, 초과면 S02를 복귀 지점으로 결정
                    
                    # 복귀할 때는 is_return=True를 넘겨주어 Y=3.0 앞쪽에서는 유클리드 거리로 대각선 복귀하도록 호출
                    t, d = self.calculate_travel_time(target_station, is_return=True) 
                    
                    self.warehouse.move_log.append({ # 분석을 위해 로봇의 스테이션 복귀 이력을 창고 로그에 추가
                        'time': self.env.now, # 현재 시뮬레이션 시간 기록
                        'ctu_id': self.id, # 복귀하는 로봇 ID 기록
                        'from_x': self.pos[0], 'from_y': self.pos[1], 'from_z': self.pos[2], # 마지막 피킹 지점 좌표 기록
                        'to_x': target_station[0], 'to_y': target_station[1], 'to_z': target_station[2], # 목적지 스테이션 좌표 기록
                        'item_id': 'RETURN', # 피킹 작업이 아닌 스테이션 복귀임을 의미하는 태그 삽입
                        'travel_time': t, 'travel_distance': d, # 복귀 구간 소요 시간 및 거리 기록
                        'dispatch_rule': self.dispatch_rule, # 당시 적용된 디스패칭 룰 기록
                    })
                    
                    yield self.env.timeout(t) # 복귀 시간만큼 시뮬레이션 시간 지연
                    self.pos = target_station # 복귀가 완료되었으므로 로봇 위치를 스테이션 좌표로 갱신
                    self.total_distance += d # 복귀 이동 거리 누적 더하기
                
                    # 모든 작업과 복귀를 마쳤으므로 점유했던 통로들을 깨끗하게 비워 해제
                    self.current_batch_aisles.clear()
                    self.busy = False # 로봇 상태를 다시 대기 중(False)으로 변경
                else: 
                    # 큐에 짐은 남아있으나, 다른 CTU가 모두 점유하고 있어 배치를 받지 못한 경우 1초간 대기하며 눈치보기
                    yield self.env.timeout(1) 
            else: # 아예 창고 큐 전체에 남은 할당 작업이 없는 경우
                yield self.env.timeout(1) # 무한 루프 과부하를 막기 위해 1초 대기 후 다시 작업 확인


# ==========================================
# 창고 시스템 클래스 정의
# ==========================================
class Warehouse: # 전체 창고, 재고, 큐, 로그를 통합 관리하는 클래스
    def __init__(self, env, inventory, plan_df): # 창고 초기화 메서드
        self.env = env # SimPy 시뮬레이션 환경 저장
        self.inventory = inventory # 슬로팅 룰에 의해 생성된 {부품: 위치} 맵핑 데이터 저장
        self.plan_df = plan_df # 금일 처리해야 할 전체 출고 계획 주문 데이터 저장
        self.job_queue = [] # 실시간으로 쏟아지는 주문(부품 피킹)들이 대기하는 리스트(큐) 초기화
        
        self.dispatch_log = [] # 디스패칭 룰에 의해 배치 작업이 어떻게 생성되었는지 기록할 리스트
        self.move_log = [] # 로봇의 점대점 이동 상세 이력을 기록할 리스트
        
        self.ctus = [] # 교착(Deadlock) 방지를 위해 창고가 전체 CTU들의 객체 정보를 알고 있도록 리스트 초기화

    def apply_dispatching(self, ctu, rule): # 룰에 따라 큐에서 작업을 꺼내 로봇에 할당해주는 메서드
        if not self.job_queue: return [] # 만약 큐가 비어있다면 빈 리스트 반환하여 작업 할당 취소

        # 병목 현상 방지: 현재 작업을 호출한 본인이 아닌 다른 CTU가 작업 중인 통로(X좌표) 목록을 수집
        occupied_aisles = set()
        for c in self.ctus:
            if c.id != ctu.id and c.busy: # 자신이 아니며, 현재 작업을 수행 중인 CTU인 경우
                occupied_aisles.update(c.current_batch_aisles) # 해당 CTU가 점유 중인 통로 목록을 블랙리스트(occupied_aisles)에 병합

        # 전체 대기 큐의 작업들 중에서 점유(블랙리스트)되지 않은 통로에 위치한 작업들만 골라 가용 풀을 생성
        available_jobs = [j for j in self.job_queue if j['pos'][0] not in occupied_aisles]

        # 만약 겹치지 않는 작업이 단 하나도 없다면 당장 할당을 포기하고 빈 리스트를 반환하여 안전하게 대기 유도
        if not available_jobs:
            return [] 

        # 1. Nearest
        if rule == 'nearest': # 디스패칭 룰이 가장 가까운 작업 우선(Nearest)인 경우
            # 소요 시간(Time) 기준으로 가장 빨리 도착할 수 있는 작업 정렬
            available_jobs.sort(key=lambda j: get_estimated_travel_time(ctu.pos, j['pos'])) 
            batch = [available_jobs.pop(0)] # 정렬된 가용 풀에서 제일 앞(가장 가까운) 작업을 Seed(기준 작업)로 뽑아 배치에 첫 번째로 담음
            
            if available_jobs: # 가용 풀에 남은 작업이 더 있다면
                available_jobs.sort(key=lambda j: get_estimated_travel_time(batch[0]['pos'], j['pos'])) # 방금 뽑은 Seed 작업 위치를 기준으로 가까운 순으로 재정렬
                extra_count = min(len(available_jobs), TOTE_CAPACITY - 1) # 남은 가용 작업 수와 로봇 여유 적재량(7-1=6) 중 작은 값을 추가 작업 수로 확정
                batch.extend(available_jobs[:extra_count]) # Seed 주변의 추가 가용 작업들을 배치 리스트에 병합
                
            # 원본 job_queue에서 최종 할당된 배치의 작업들만 찾아내어 안전하게 삭제
            for j in batch:
                self.job_queue.remove(j)
                
            self.dispatch_log.append({ # 배치 구성 완료 후 배치 로그 기록
                'time': ctu.env.now, 'ctu_id': ctu.id, 'rule': 'nearest', # 시간, 로봇, 적용 룰 기록
                'seed_item': batch[0]['item_id'], 'batch_items': [j['item_id'] for j in batch], # 기준 부품과 전체 배치 부품 목록 기록
                'batch_size': len(batch), # 이 배치의 총 작업 수 기록
            })
            return batch # 완성된 묶음(배치) 반환

        # 2. Density
        elif rule == 'density': # 디스패칭 룰이 작업 밀집도 우선(Density)인 경우
            K = min(10, len(available_jobs)) # 가용 작업 중 로봇에서 가장 가까운 상위 10개만 Seed 후보로 추림
            candidate_seeds = sorted(available_jobs, key=lambda j: get_estimated_travel_time(ctu.pos, j['pos']))[:K] 
            
            best_seed = None # 최적의 Seed 작업을 저장할 변수 초기화
            max_density = -1 # 가장 높은 밀집도 점수를 저장할 변수 초기화
            r_y = 2.65 # 같은 통로 내에서 밀집도를 계산할 검색 반경(2.65m) 설정
            
            for seed in candidate_seeds: # 후보 Seed 10개 각각에 대해 주변 작업 밀집도 조사
                density = 0 # 현재 평가 중인 Seed의 밀집도 카운터 0으로 시작
                for job in available_jobs: # 밀집도를 계산할 때도 다른 로봇이 점유한 통로의 작업은 철저히 배제
                    if job['pos'][0] == seed['pos'][0]: # 평가 대상 작업이 Seed 작업과 같은 통로(X좌표)에 있는지 확인
                        dy = abs(job['pos'][1] - seed['pos'][1]) # 같은 통로라면 Y좌표(깊이) 차이 계산
                        if dy <= r_y: # Y축 거리가 반경(2.65m) 이내라면
                            density += 1 # 주변 작업이므로 밀집도 카운트 1 증가
                
                if density > max_density: # 현재 계산한 밀집도가 역대 최대 밀집도보다 높다면
                    max_density = density # 최대 밀집도 갱신
                    best_seed = seed # 최적 Seed를 현재 Seed로 변경
                    
            available_jobs.sort(key=lambda j: get_estimated_travel_time(best_seed['pos'], j['pos'])) # 찾아낸 최적 밀집 Seed를 기준으로 가용 풀을 정렬
            
            batch_size = min(len(available_jobs), TOTE_CAPACITY) # 가용 남은 물량과 로봇 최대 용량 중 묶을 크기 결정
            best_batch = available_jobs[:batch_size] # 정렬된 가용 큐에서 위에서 결정한 개수만큼 묶음 생성
            
            # 원본 전체 job_queue에서 배치로 뽑혀나간 작업들을 삭제
            for j in best_batch:
                self.job_queue.remove(j)
            
            self.dispatch_log.append({ # 밀집도 기반 배치 생성 로그 기록
                'time': ctu.env.now, 'ctu_id': ctu.id, 'rule': 'density', # 시간, 로봇 ID, 룰 종류 기록
                'seed_item': best_seed['item_id'], 'batch_items': [j['item_id'] for j in best_batch], # 기준 부품 및 배치 내 부품 리스트 기록
                'batch_size': len(best_batch), 'local_density': max_density, # 배치 크기 및 해당 지역의 계산된 밀집도 점수 기록
            })
            return best_batch # 생성된 밀집 배치 반환

        # 3. Load-balance
        elif rule == 'load_balance': # 디스패칭 룰이 최단 소요 시간 예측(Load-balance)인 경우
            K = min(10, len(available_jobs)) # 가용 풀 작업 중 가까운 상위 10개를 Seed 후보로 추출
            nearest_jobs = sorted(available_jobs, key=lambda j: get_estimated_travel_time(ctu.pos, j['pos'])) 
            candidate_seeds = nearest_jobs[:K] 
            
            best_batch = [] # 최단 시간으로 처리될 최적의 배치를 저장할 리스트 초기화
            min_eta = float('inf') # 최소 예상 소요 시간(ETA)을 무한대로 초기화 (비교 갱신용)
            best_seed_item = None # 최적 배치에 사용된 Seed 부품 코드를 저장할 변수 초기화
            
            for seed in candidate_seeds: # 10개의 각 후보 Seed를 가상으로 배치로 묶어봄
                # 임시 큐를 가용 작업(available_jobs)만을 대상으로 Seed 기준 가까운 순 정렬
                temp_queue = sorted(available_jobs, key=lambda j: get_estimated_travel_time(seed['pos'], j['pos'])) 
                temp_batch = temp_queue[:TOTE_CAPACITY] # 임시 가용 큐에서 최대 용량만큼 가상 묶음 구성
                
                eta = 0 # 이 가상 배치를 처리할 때의 예상 소요 시간(ETA) 0으로 초기화
                current_pos = ctu.pos  # 로봇의 이동을 시뮬레이션하기 위해 가상 위치를 현재 로봇 위치로 설정
                
                for job in temp_batch: # 가상 배치의 작업들을 순서대로 가상 방문
                    x1, y1, z1 = current_pos # 가상 현재 좌표 언패킹
                    x2, y2, z2 = job['pos'] # 가상 이동 목표 좌표 언패킹
                    
                    if x1 == x2: # 통로가 같은지 확인
                        xy_dist = abs(y1 - y2) # 평면 직진 거리 계산
                    else: # 통로가 다르면
                        xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y) # 교차 통로 우회 거리 계산
                        
                    dz = abs(z1 - z2) # 리프트 이동 거리 계산
                    time_to_job = max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z) # 현재 지점에서 목표 작업까지의 시간 계산
                    eta += time_to_job # 계산된 구간 시간을 가상 총 소요 시간(ETA)에 누적
                    current_pos = job['pos']  # 가상 로봇 위치를 방금 계산한 작업 지점으로 갱신
                
                # --- 스테이션 복귀 시간 ETA에 반영 ---
                last_x, last_y, last_z = current_pos
                target_station = STATIONS['S01'] if last_x <= 6.0 else STATIONS['S02']
                
                # 복귀 경로 이동 거리 산출 (is_return=True 방식 적용)
                dist_to_cross_aisle = abs(last_y - CROSS_AISLE_Y)
                euclidean_dist_to_station = math.sqrt((last_x - target_station[0])**2 + (CROSS_AISLE_Y - target_station[1])**2)
                return_xy_dist = dist_to_cross_aisle + euclidean_dist_to_station
                return_dz = abs(last_z - target_station[2])
                
                return_time = max(return_xy_dist / CTU_SPEED_XY, return_dz / CTU_SPEED_Z)
                eta += return_time # 총 소요 시간에 복귀 시간을 합산해야 진짜 Makespan 산출이 가능
                # -----------------------------------------------------------------
                
                if eta < min_eta: # 이 가상 배치의 총 예상 시간이 이전 최소값보다 더 짧다면
                    min_eta = eta # 최소 예상 시간 갱신
                    best_batch = temp_batch # 최적의 배치를 방금 만든 가상 배치로 교체
                    best_seed_item = seed['item_id'] # 그때 사용된 Seed 부품명 저장
            
            # 최종 확정된 배치의 부품들을 진짜 대기 큐(job_queue)에서 안전하게 지우기
            for job in best_batch: 
                self.job_queue.remove(job) 
                
            self.dispatch_log.append({ # 로드 밸런스 기반 배치 생성 로그 기록
                'time': ctu.env.now, 'ctu_id': ctu.id, 'rule': 'load_balance', # 기본 메타데이터 기록
                'seed_item': best_seed_item, 'batch_items': [j['item_id'] for j in best_batch], # Seed 정보와 할당 부품 리스트 기록
                'batch_size': len(best_batch), 'eta': min_eta, # 배치 크기와 시뮬레이션 전 예측했던 예상 종료 시간(ETA) 기록
            })
            return best_batch # 가장 빨리 끝나는 최적 배치 반환

#슬로팅 로직 수정 
def generate_fixed_inventory(rule_type, part_list, weekly_prod_plan, bom_df, outbound_plan, seed=1234):
    """
    교수님 피드백 반영: 
    1. 제품별 생산 계획(물량)을 고려
    2. kitting 시 같은 제품의 부품들을 물리적으로 인접하게 배치
    """
    random.seed(seed)
    inventory = {}
    
    # 1. 제품별 주간 생산 물량 계산
    prod_vol = weekly_prod_plan.groupby('제품코드').size().to_dict()
    # 물량이 많은 순서대로 제품 정렬
    sorted_products = sorted(prod_vol.keys(), key=lambda x: prod_vol.get(x, 0), reverse=True)

    # 2. 가용 슬롯 생성 및 정렬 (스테이션에서 가까운 순)
    y_positions = np.linspace(RACK_START_Y, RACK_END_Y, num=24)
    candidate_slots = []
    for x in AISLE_X:
        for y in y_positions:
            for z in SHELF_HEIGHTS:
                candidate_slots.append((x, round(y,2), z))

    def est_time_to_station(pos): # 각 슬롯에서 스테이션까지의 예상 접근 시간을 구하는 내부 함수
        x, y, z = pos
        min_time = float('inf')
        for st in STATIONS.values():
            dist = abs(y - CROSS_AISLE_Y) + math.sqrt((x - st[0])**2 + (CROSS_AISLE_Y - st[1])**2)
            t = max(dist / CTU_SPEED_XY, abs(z - st[2]) / CTU_SPEED_Z)
            if t < min_time: min_time = t
        return min_time

    candidate_slots.sort(key=est_time_to_station) # 슬롯들을 스테이션 도달 시간이 짧은(가까운) 순으로 미리 정렬

    # 3. 전략별 부품 배치 순서 결정 (Parts Ordering)
    parts_order = []
    used_parts = set()

    # generate_fixed_inventory 함수 내부의 Kitting 전략 부분 수정 제안
    # --- generate_fixed_inventory 함수 내부의 kitting 조건절 수정 ---
    if rule_type == "kitting":
        # 1. 함께 출고되는 부품 쌍(Pair)의 빈도 계산
        pair_counter = Counter()
        for _, group in outbound_plan.groupby('No.'):
            parts = sorted(group['부품코드'].unique())
            if len(parts) > 1:
                for pair in combinations(parts, 2):
                    pair_counter[pair] += 1
        
        # 2. 빈도가 높은 쌍부터 확인하며 '묶음(Cluster)' 생성
        # 조건: 한 묶음은 최소 2개(쌍으로 시작), 최대 8개까지
        clusters = []
        assigned_parts = set()
        
        # 빈도순으로 정렬된 부품 쌍들
        sorted_pairs = [pair for pair, count in pair_counter.most_common()]
        
        for p1, p2 in sorted_pairs:
            # 케이스 1: 둘 다 아직 어느 묶음에도 속하지 않음 -> 새로운 묶음 생성 (최소 2개 만족)
            if p1 not in assigned_parts and p2 not in assigned_parts:
                clusters.append([p1, p2])
                assigned_parts.add(p1)
                assigned_parts.add(p2)
            
            # 케이스 2: p1은 이미 묶음에 있고 p2는 없음 -> p1이 있는 묶음에 p2 추가 (최대 8개 체크)
            elif p1 in assigned_parts and p2 not in assigned_parts:
                for cluster in clusters:
                    if p1 in cluster and len(cluster) < 8:
                        cluster.append(p2)
                        assigned_parts.add(p2)
                        break
            
            # 케이스 3: p2는 이미 묶음에 있고 p1은 없음 -> p2가 있는 묶음에 p1 추가 (최대 8개 체크)
            elif p2 in assigned_parts and p1 not in assigned_parts:
                for cluster in clusters:
                    if p2 in cluster and len(cluster) < 8:
                        cluster.append(p1)
                        assigned_parts.add(p1)
                        break
        
        # 3. 생성된 묶음들을 순서대로 parts_order에 추가
        # (묶음 내의 부품들은 parts_order에서 나란히 배치되므로, 결과적으로 창고에서도 가깝게 배치됨)
        parts_order = []
        for cluster in clusters:
            parts_order.extend(cluster)
            
        # 4. 묶이지 않은 나머지 부품들(단독 출고 등)을 뒤에 추가
        remaining_parts = [p for p in part_list if p not in assigned_parts]
        parts_order.extend(remaining_parts)

    elif rule_type == "characteristic":
        # 생산 물량이 가장 많은 Top 제품의 부품들을 전진 배치
        if sorted_products:
            top_p = sorted_products[0]
            top_parts = set(bom_df[bom_df['product_code'] == top_p]['part_code'].unique())
            parts_order = [p for p in part_list if p in top_parts] + \
                          [p for p in part_list if p not in top_parts]
        else:
            parts_order = list(part_list)

    elif rule_type == "abc":
        # freq_time과 유사하게 정렬 후 등급 개념 부여 (여기서는 정렬만 수행)
        # 각 부품이 포함된 제품들의 전체 생산 물량(vol)을 합산하여 부품의 총 수요(part_demand)를 계산합니다.
        part_demand = {}
        for p_code, vol in prod_vol.items(): # 모든 제품의 주간 생산 물량을 순회
            p_parts = bom_df[bom_df['product_code'] == p_code]['part_code'].unique() # 해당 제품에 들어가는 부품 목록 추출
            for p in p_parts:
                part_demand[p] = part_demand.get(p, 0) + vol # 각 부품에 생산 물량 가중치를 누적
        # 수요(생산 물량 합계)가 높은 부품일수록 리스트의 앞쪽에 오도록 내림차순(reverse=True) 정렬합니다.
        parts_order = sorted(part_list, key=lambda x: part_demand.get(x, 0), reverse=True)

    else: # random
        # 슬로팅 규칙이 무작위(Random)인 경우
        parts_order = list(part_list) # 전체 부품 리스트를 복사한 뒤
        random.shuffle(parts_order)   # 부품의 순서를 완전히 무작위로 섞습니다. (나중에 슬롯에 차례대로 배정되므로 무작위 배치가 됨)

    # 4. 부품과 슬롯 매핑 (1:1)
    for part, slot in zip(parts_order, candidate_slots):
        inventory[part] = list(slot)

    return inventory

# ==========================================
# 하루 단위 시뮬레이션 실행
# ==========================================
def run_simulation(date, inventory, dispatch_rule, date_plan): # 특정 일자의 작업을 시뮬레이션하는 메인 구동 함수
    env = simpy.Environment() # 이산 사건 시뮬레이션의 근간이 되는 SimPy 환경 객체 생성
    warehouse = Warehouse(env, inventory, date_plan) # 위에서 만든 재고와 일일 계획을 바탕으로 창고 객체 인스턴스화
    
    ctus = [CTU(env, i, warehouse, dispatch_rule) for i in range(2)] # 창고 내에서 주어진 디스패칭 룰에 따라 CTU 객체 2대 생성
    warehouse.ctus = ctus # 창고 시스템이 전체 로봇의 위치(점유 통로)를 알 수 있도록 방금 생성한 CTU 리스트를 창고 속성에 주입
    
    def job_gen(env, wh): # 계획에 맞춰 주문 발생을 모사하는 내부 제너레이터 함수
        # 일관된 결과를 위해 각 날짜마다 시드를 일정하게 맞추어 주문 발생 타이밍 통제
        random.seed(hash(date) % 10000) 
        
        for _, group in wh.plan_df.groupby('No.'): # 일일 주문 계획을 주문번호(No.) 별로 묶어 순회
            yield env.timeout(random.uniform(2, 5))  # 각 주문이 시스템에 인입되는 데 걸리는 무작위 시간(2~5초) 간격 부여
            for _, row in group.iterrows(): # 하나의 주문 묶음 내 포함된 부품 리스트 순회
                p_id = row['부품코드'] # 작업해야 할 부품 코드 추출
                if p_id in wh.inventory: # 재고 맵에 이 부품의 위치가 존재하는지 안전 검사
                    wh.job_queue.append({'item_id': p_id, 'pos': wh.inventory[p_id], 'time': env.now}) # 창고 대기 큐에 이 부품의 위치, 주문 발생 시간 기록

    def monitor(env, warehouse, gen_proc, ctus): # 시뮬레이션 종료 시점을 감시하는 내부 함수
        yield gen_proc # 주문 발생 제너레이터(job_gen)가 모든 주문 인입을 끝낼 때까지 대기
        while warehouse.job_queue or any(c.busy for c in ctus): # 주문 생성이 끝난 후에도 창고 큐에 작업이 남아있거나, 아직 작업 중인 로봇이 있다면
            yield env.timeout(1)  # 1초씩 시간을 흐르게 하며 모니터링 반복 루프 지속 (모두 종료되면 루프 탈출)

    gen_proc = env.process(job_gen(env, warehouse)) # 주문 발생 제너레이터를 SimPy 프로세스로 등록
    env.run(until=env.process(monitor(env, warehouse, gen_proc, ctus))) # 모니터 함수(프로세스)가 종료될 때까지 전체 시뮬레이션 엔진 가동
    
    return { # 하루치 시뮬레이션이 끝난 뒤 분석용 데이터 반환
        'Date': date,  # 시뮬레이션 실행 날짜
        'Time': round(env.now, 1),  # 시뮬레이션의 마지막 순간(총 작업 소요 시간) 반환
        'Distance': round(sum(c.total_distance for c in ctus), 1), # 두 로봇의 이동 거리 총합산 반환
        'Slotting': None,      # 메인에서 채워넣을 예정
        'Dispatching': None,   # 메인에서 채워넣을 예정
        'Move_log': pd.DataFrame(warehouse.move_log), # 쌓인 로봇 이동 이력 로그를 데이터프레임으로 변환하여 반환
        'Dispatch_log': pd.DataFrame(warehouse.dispatch_log), # 쌓인 배치 할당 로그 데이터프레임 변환 반환
    }

# ==========================================
# 개별 알고리즘 조합 통합 시각화 
# ==========================================
def visualize_rule_combination(move_all_df, slot_rule, disp_rule): # 호출 시마다 특정 슬로팅/디스패칭 조합을 시각화하도록 매개변수 추가
    """
    입력받은 슬로팅(slot_rule)과 디스패칭(disp_rule) 조합의 시뮬레이션 결과를 
    3가지 방식(3D Scatter, 2D Trajectory, Gantt Chart)으로 시각화합니다.
    (기존 Kitting + Load_balance 전용 함수를 모든 조합에 범용적으로 쓸 수 있게 확장)
    """
    # 1. 데이터 필터링: 입력받은 조합(slot_rule, disp_rule) 추출
    target_df = move_all_df[(move_all_df['Slotting'] == slot_rule) &  # 전체 이력에서 슬로팅 룰 조건 필터링
                            (move_all_df['Dispatching'] == disp_rule)].copy() # 디스패칭 룰 조건까지 AND 결합하여 복사
    
    if target_df.empty: # 해당 조합의 데이터가 비어있다면
        print(f"해당 조건({slot_rule} + {disp_rule})의 데이터가 없습니다.") # 에러 메시지 출력 후 스킵
        return # 함수 종료
        
    # 시각화가 너무 복잡해지는 것을 막기 위해 첫 번째 시뮬레이션 날짜만 선택
    sample_date = target_df['Date'].iloc[0] # 타겟 데이터프레임에서 가장 첫 번째 줄의 시뮬레이션 날짜 문자열 추출
    df_day = target_df[target_df['Date'] == sample_date].copy() # 전체 데이터 중 방금 추출한 단일 날짜의 결과만 필터링하여 복사
    
    print(f"\n=== [시각화 진행 중: Date {sample_date} | {slot_rule} × {disp_rule}] ===") # 현재 시각화하는 조합과 날짜 정보를 터미널에 명확히 안내

    # ---------------------------------------------------------
    # [1] 3D Scatter Plot: 부품 피킹 위치 분포
    # ---------------------------------------------------------
    fig1 = plt.figure(figsize=(10, 8)) # 3D 산점도를 그리기 위해 가로 10, 세로 8 크기의 새 피규어 생성
    ax1 = fig1.add_subplot(111, projection='3d') # 위 피규어의 첫 번째 영역에 3D 투영(projection)을 활성화한 축 객체 생성
    picks = df_day[df_day['item_id'] != 'RETURN'] # 스테이션 복귀는 제외하고 실제 부품 좌표 데이터만 추출
    pick_counts = picks.groupby(['to_x', 'to_y', 'to_z', 'item_id']).size().reset_index(name='count') # 동일한 X,Y,Z 위치에 피킹이 몇 번 일어났는지 집계하여 count 열로 명명
    
    scatter = ax1.scatter(pick_counts['to_x'], pick_counts['to_y'], pick_counts['to_z'],  # 추출한 x, y, z 좌표 리스트를 기반으로 3D 산점도 그리기
                          s=pick_counts['count']*50, c=pick_counts['to_z'], cmap='viridis', alpha=0.7) # 빈도(count)가 클수록 점 크기(s) 확대, 층수(Z)에 따라 색상 분리, 반투명도 지정
    
    ax1.set_title(f'① 3D Scatter: 피킹 위치 분포 ({slot_rule} × {disp_rule})', fontsize=14, fontweight='bold') # 3D 플롯 제목에 현재 적용 중인 룰 변수를 동적 삽입
    ax1.set_xlabel('통로 X (Aisle)') # 3D 플롯 X축 레이블 지정
    ax1.set_ylabel('랙 깊이 Y') # 3D 플롯 Y축 레이블 지정
    ax1.set_zlabel('선반 높이 Z') # 3D 플롯 높이 Z축 레이블 지정
    fig1.colorbar(scatter, ax=ax1, label='Z축 높이 (층수)') # Z 높이에 따른 범례 역할을 하는 컬러바를 우측에 부착
    plt.tight_layout() # 요소 간 겹침 방지 여백 조정
    plt.show() # 3D 산점도 화면 출력

    # ---------------------------------------------------------
    # [2] 2D Trajectory Map: 로봇 이동 궤적 밀집도 (Heatmap 스타일 렌더링)
    # ---------------------------------------------------------
    fig2, ax2 = plt.subplots(figsize=(12, 8)) # 로봇 궤적을 그릴 12x8 사이즈 피규어와 2D 축 생성
    
    # 랙 및 스테이션 배경 그리기: 기획안 도면(R01 ~ R12)과 완전히 동일한 X좌표와 폭(Width)으로 백투백(Back-to-Back) 랙 덩어리들을 생성
    rack_spans = [(0, 0.6), (1.8, 1.2), (4.2, 1.2), (6.6, 1.2), (9.0, 1.2), (11.4, 1.2), (13.8, 0.6)] # 구조: (단일 0.6m) - (양면 1.2m) * 5 - (단일 0.6m), 통로 간격은 1.2m씩
    for start_x, width in rack_spans: # 랙 덩어리 좌표 배열을 순회
        ax2.add_patch(patches.Rectangle((start_x, RACK_START_Y), width, RACK_END_Y - RACK_START_Y, color='lightgray')) # 도면에 맞게 회색 랙 덩어리 배치
    
    for x in AISLE_X: # 통로 라벨 추가
        ax2.text(x, RACK_END_Y + 0.5, f"Aisle {x}", ha='center', fontsize=9) # 각 통로 위에 번호 텍스트 부착
        
    ax2.axhline(CROSS_AISLE_Y, color='black', linestyle='--', alpha=0.3, label='교차 통로 (Cross Aisle)') # 로봇이 교차하여 지나가는 Y축(2.0) 위치에 검은 점선 가이드라인 표시
    
    ax2.add_patch(patches.Rectangle((2.6, 0), 2.0, 1.0, color='lightgreen', label='Station (S01, S02)')) # 작업 스테이션 S01 위치에 연녹색 박스 도형 생성 및 배치
    ax2.add_patch(patches.Rectangle((9.8, 0), 2.0, 1.0, color='lightgreen')) # 작업 스테이션 S02 위치에 도형 배치
    ax2.text(3.6, 0.5, 'S01', ha='center', va='center', fontweight='bold') # S01 박스 정중앙에 'S01' 텍스트 레이블 추가
    ax2.text(10.8, 0.5, 'S02', ha='center', va='center', fontweight='bold') # S02 박스 내부 텍스트 레이블 추가

    # ----- 핵심 로직: 궤적을 5cm 단위의 점(Point)으로 쪼개 통행량 카운트 -----
    traffic_counts = {} # 특정 (x, y) 좌표를 지날 때마다 누적 횟수를 저장할 빈 딕셔너리 생성

    def add_line_segment(sx, sy, ex, ey): # 시뮬레이션의 선형 궤적을 점 단위로 분해하는 내부 함수
        """시작점과 끝점을 받아 그 사이의 경로를 5cm 간격의 점으로 변환 후 카운트 증가"""
        # [수정사항] 궤적 렌더링 시에도 대각선 이동을 매끄럽게 그릴 수 있도록 맨해튼 방식이 아닌 유클리드 거리로 점의 개수를 산출
        dist = math.sqrt((sx - ex)**2 + (sy - ey)**2) 
        steps = int(dist / 0.05) + 1 # 0.05m(5cm) 해상도로 경로를 쪼개어 생성할 점의 전체 개수 계산
        for i in range(steps): # 계산된 점의 개수만큼 루프 반복
            px = sx + (ex - sx) * i / max(1, steps - 1) # i번째 점의 X 좌표 보간 계산
            py = sy + (ey - sy) * i / max(1, steps - 1) # i번째 점의 Y 좌표 보간 계산
            coord = (round(px, 2), round(py, 2)) # 계산된 점을 소수점 2자리로 정규화하여 튜플 생성
            traffic_counts[coord] = traffic_counts.get(coord, 0) + 1 # 딕셔너리에 해당 좌표 키가 있으면 값을 1 증가, 없으면 1로 초기화 (통행 빈도 누적)

    # 로봇의 모든 점대점 이동 이력 순회
    for _, row in df_day.iterrows(): # 해당 날짜 로봇의 모든 이동 이력을 한 줄씩 순회
        x1, y1 = row['from_x'], row['from_y'] # 출발점 X, Y 좌표 추출
        x2, y2 = row['to_x'], row['to_y'] # 도착점 X, Y 좌표 추출
        
        if row['item_id'] == 'RETURN': 
            # [수정사항] 복귀 궤적: 랙에서 빠져나와 교차통로(Y=3.0)까지는 직진 이동 후, 스테이션까지 유클리드(대각선) 궤적을 추가
            add_line_segment(x1, y1, x1, CROSS_AISLE_Y) # 랙에서 교차 통로로 나오기
            add_line_segment(x1, CROSS_AISLE_Y, x2, y2) # 교차 통로에서 스테이션으로 대각선 횡단
        else:
            if x1 == x2: # 출발 통로와 도착 통로가 같아 꺾을 필요가 없다면 (한 번에 직진)
                add_line_segment(x1, y1, x2, y2) # 해당 직진 구간 전체를 점으로 쪼개 카운트 등록
            else: # 다른 통로로 이동하는 경우 (교차통로 우회 3단계 맨해튼 궤적)
                add_line_segment(x1, y1, x1, CROSS_AISLE_Y)        # 1. 현재 랙 안쪽에서 교차로 지점(Y=3.0)까지 후퇴하는 직선 구간 카운트
                add_line_segment(x1, CROSS_AISLE_Y, x2, CROSS_AISLE_Y) # 2. 교차로 지점 내에서 목표 통로(X2)까지 가로로 횡단하는 구간 카운트
                add_line_segment(x2, CROSS_AISLE_Y, x2, y2)        # 3. 목표 통로 교차로에서 최종 랙 안쪽(Y2)으로 깊숙이 진입하는 직선 구간 카운트

    # 수집된 통행 빈도 데이터를 시각화를 위한 리스트로 분리
    pts_x, pts_y, pts_c = [], [], [] # X좌표 리스트, Y좌표 리스트, 빈도수(색상값) 리스트 초기화
    for (x, y), count in traffic_counts.items(): # 누적된 통행 딕셔너리를 순회
        pts_x.append(x) # 분리된 X 리스트에 추가
        pts_y.append(y) # 분리된 Y 리스트에 추가
        pts_c.append(count) # 빈도수를 리스트에 추가

    # -------------------------------------------------------------------
    # Scatter Plot을 활용해 선 대신 연속된 점들로 그라데이션(Heatmap) 렌더링
    # cmap='Reds': 한 번 지나가면 연한 핑크, 여러 번 지나가 병목인 곳은 진한 빨강
    # -------------------------------------------------------------------
    sc = ax2.scatter(pts_x, pts_y, c=pts_c, cmap='Reds', s=15, zorder=3, alpha=0.9) # 추출한 리스트를 산점도로 그리되 빈도(pts_c)에 따라 빨간색 농도를 다르게 적용
    
    # 우측에 컬러바를 부착하여 어떤 색상이 얼마나 많은 횟수를 의미하는지 범례 제공
    cbar = fig2.colorbar(sc, ax=ax2) # 방금 그린 산점도 색상 기준을 축으로 삼아 컬러바 생성
    cbar.set_label('로봇 누적 통행 빈도 (Traffic Intensity)') # 컬러바 옆 설명 텍스트 부착

    ax2.set_title(f'② 2D Trajectory Map: 로봇 통행 밀집도 궤적 ({slot_rule} × {disp_rule})', fontsize=14, fontweight='bold') # 2D 궤적 맵 제목에 현재 적용 중인 조합 변수명 동적 표시
    ax2.set_xlabel('X 좌표 (m)') # 궤적 맵 X축 물리적 단위(미터) 라벨
    ax2.set_ylabel('Y 좌표 (m)') # 궤적 맵 Y축 라벨
    ax2.set_xlim(-1, 15.5) # 맨 우측 랙이 잘리지 않도록 X축 오른쪽 여백 확대
    ax2.set_ylim(-1, 17) # 하단 스테이션 영역과 랙 상단 여백 표시를 위해 Y 범위 -1 ~ 17 확보
    
    # 기존 범례를 깔끔하게 유지하기 위한 정리 로직
    handles, labels = ax2.get_legend_handles_labels() # 플롯에 추가된 모든 막대 종류의 범례 핸들과 라벨 텍스트 수집
    by_label = dict(zip(labels, handles)) # 중복 라벨을 하나로 합치기 위해 딕셔너리 구조 활용
    ax2.legend(by_label.values(), by_label.keys(), loc='upper left') # 좌측 상단에 정돈된 범례 표시
    
    plt.tight_layout() # 간격 자동 조절
    plt.show() # 2D 궤적 맵 화면 출력

    # ---------------------------------------------------------
    # [3] Gantt Chart: CTU별 작업 시간 및 유휴 시간 확인
    # ---------------------------------------------------------
    fig3, ax3 = plt.subplots(figsize=(12, 4)) # 가로로 긴 형태(12x4)의 간트 차트 피규어 생성
    ctu_tasks = {0: [], 1: []} # 각 로봇의 (작업 시작 시점, 작업 소요 시간) 정보 튜플을 담을 빈 리스트 딕셔너리
    ctu_returns = {0: [], 1: []} # 복귀 구간도 별도의 색상으로 구분하기 위해 (시작, 소요) 튜플을 담을 딕셔너리
    
    for _, row in df_day.iterrows(): # 모든 이동 이력을 시간순으로 순회
        end_time = row['time'] # 해당 구간 이동이 끝난 완료 시간
        duration = row['travel_time'] # 해당 구간 이동에 걸린 소요 시간(초)
        start_time = end_time - duration # 이동 완료 시점에서 소요 시간을 빼 이동이 시작된 출발 시간 역산
        
        if row['item_id'] == 'RETURN': # 만약 이동 목적이 작업이 아닌 스테이션 복귀라면
            ctu_returns[row['ctu_id']].append((start_time, duration)) # 복귀 리스트에 시작, 지속시간 튜플 저장
        else: # 일반 부품 피킹 이동이라면
            ctu_tasks[row['ctu_id']].append((start_time, duration)) # 작업 리스트에 시작, 지속시간 저장
            
    ax3.broken_barh(ctu_tasks[0], (5, 8), facecolors='tab:blue', label='CTU 0 (피킹)') # CTU 0번 작업 리스트의 막대들을 Y축 기준 5~13 위치에 파란색으로 그림
    ax3.broken_barh(ctu_returns[0], (5, 8), facecolors='lightblue', label='CTU 0 (복귀)') # CTU 0번 복귀 구간 막대를 같은 Y축 라인에 연파랑으로 겹쳐서 표시
    ax3.broken_barh(ctu_tasks[1], (15, 8), facecolors='tab:red', label='CTU 1 (피킹)') # CTU 1번 작업 막대를 Y축 15~23 윗줄에 빨강으로 배치
    ax3.broken_barh(ctu_returns[1], (15, 8), facecolors='lightcoral', label='CTU 1 (복귀)') # CTU 1번 복귀 구간 연빨강 처리
    
    ax3.set_ylim(0, 30) # Y축의 범위를 0부터 30까지 넉넉하게 설정하여 두 줄의 막대가 중앙에 잘 보이도록 여백 확보
    ax3.set_xlim(0, df_day['time'].max() + 10) # X축(시간)을 0초부터 가장 마지막 기록된 종료 시간의 +10초 여유분까지 설정
    ax3.set_yticks([9, 19]) # Y축 좌측 라벨이 막대의 정중앙(5+4, 15+4)을 가리키도록 틱 위치 9, 19 지정
    ax3.set_yticklabels(['CTU 0', 'CTU 1']) # 지정된 틱 위치에 로봇 이름 텍스트 표시
    ax3.set_xlabel('시뮬레이션 시간 (초)') # X축이 시간 흐름임을 명시
    ax3.set_title(f'③ Gantt Chart: CTU 작업 흐름 ({slot_rule} × {disp_rule})', fontsize=14, fontweight='bold') # 간트 차트 제목에 조합 변수명 동적 표시
    
    handles, labels = ax3.get_legend_handles_labels() # 플롯에 추가된 모든 막대 종류의 범례 핸들과 라벨 텍스트 수집
    by_label = dict(zip(labels, handles)) # 중복 라벨을 하나로 합치기 위해 딕셔너리 구조 활용 (같은 이름이면 덮어씀)
    ax3.legend(by_label.values(), by_label.keys(), loc='upper right') # 중복 제거된 유니크한 범례를 우측 상단에 박스로 표시
    
    plt.tight_layout() # 차트 여백 최적화
    plt.show() # 최종 간트 차트 팝업 출력

# ==========================================
# 주 단위 런타임(소요 시간) 시각화 표 생성 함수 [수정]
# ==========================================
def report_weekly_scoreboard(res_df):
    """모든 조합의 일별 런타임을 주 단위로 합산하여 히트맵(Heatmap) 형태의 시각화 표로 출력합니다."""
    
    # 1. Date 컬럼을 datetime 타입으로 확실히 변환 (주차별 그룹화를 위해 필요)
    res_df['Date'] = pd.to_datetime(res_df['Date'])
    
    # ISO 캘린더 기준으로 연도와 주차(Week)를 추출하여 라벨용 파생 변수(Week) 생성 (예: '2026년 01주차')
    res_df['Week'] = res_df['Date'].dt.strftime('%G년 %V주차')
    
    # 시각화 표의 Y축(행)에 표시될 이름 구분을 위해 슬로팅 룰과 디스패칭 룰을 합친 조합(Combo) 파생 변수 생성
    res_df['Combo'] = res_df['Slotting'] + " + " + res_df['Dispatching']
    
    # 2. 주 단위 런타임 피벗 테이블 생성 
    # - index(행)에는 알고리즘 조합명을, columns(열)에는 계산된 주차 라벨을 배치
    # - values에는 기존 시뮬레이션 소요 시간(Time)을 넣되, aggfunc='sum'을 통해 해당 주차의 모든 일자 소요 시간을 합산
    weekly_pivot = res_df.pivot_table(index='Combo', columns='Week', values='Time', aggfunc='sum').round(1)
    
    # 3. 히트맵(Heatmap) 시각화 표 출력 영역 설정 (가로 14, 세로 8 크기)
    plt.figure(figsize=(14, 8))
    
    # Seaborn 히트맵 생성:
    # - annot=True를 통해 표의 각 셀 안에 실제 합산된 런타임 숫자(초)를 텍스트로 바로 표시
    # - fmt=".1f"를 통해 소수점 첫째 자리까지만 깔끔하게 출력
    # - cmap="YlOrRd" 설정으로 숫자가 클수록(오래 걸릴수록) 노랑 -> 주황 -> 빨강으로 직관적인 색상 변화 부여
    sns.heatmap(weekly_pivot, annot=True, fmt=".1f", cmap="YlOrRd", 
                linewidths=0.5, cbar_kws={'label': '주간 총 소요 시간 (초)'})
    
    # 시각화 표의 제목 및 축 레이블 설정
    plt.title('알고리즘 조합별 주간 시뮬레이션 총 소요 시간 (단위: 초)', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('생산 주차 (Week)', fontsize=12)
    plt.ylabel('알고리즘 조합 (Slotting + Dispatching)', fontsize=12)
    
    # Y축 라벨(알고리즘 조합명) 글자가 기본적으로 세로로 누워있는 것을 막기 위해 가로(0도)로 회전
    plt.yticks(rotation=0) 
    
    # 범례와 표가 겹치지 않도록 전체적인 간격 자동 최적화
    plt.tight_layout()
    # 화면에 실제 시각화 표 팝업 출력
    plt.show()

# ==========================================
# main 함수 (수정)
# ==========================================
def main():
    # 1. 데이터 파일 읽기 (주문 계획, 생산 계획, 부품 BOM 구조)
    outbound_plan = pd.read_csv('outbound_plan.csv')
    production_plan = pd.read_csv('production_plan.csv')
    bom_df = pd.read_excel('BOM.xlsx')

    # 일자별 그룹화를 위해 날짜 데이터를 문자열이나 타임스탬프에서 순수 Date(날짜) 객체로 통일
    outbound_plan['생산일자'] = pd.to_datetime(outbound_plan['생산일자']).dt.date
    production_plan['생산일자'] = pd.to_datetime(production_plan['생산일자']).dt.date
    
    # 시뮬레이션을 돌려야 할 고유한 생산 일자 리스트를 정렬하여 추출
    unique_dates = sorted(outbound_plan['생산일자'].unique())
    # 출고 계획에 존재하는 모든 종류의 부품 코드 목록 확보
    part_list = outbound_plan['부품코드'].unique().tolist() 
    
    # 3. 실험할 알고리즘 조합 정의 (총 4개 슬로팅 * 3개 디스패칭 = 12개 조합 생성)
    slotting_rules = ['random', 'abc', 'characteristic', 'kitting']
    dispatch_rules = ['nearest', 'density', 'load_balance']

    # 시뮬레이션 결과를 누적 저장할 빈 리스트들 초기화
    all_results_summary = []
    move_logs_all = []
    dispatch_logs_all = []

    print(f"총 {len(unique_dates)}일치 데이터에 대해 시뮬레이션을 시작합니다. (총 12개 조합)")

    # 4. 시뮬레이션 실행 2중 루프 (슬로팅 룰 4종을 순회하며 내부에 디스패칭 룰 3종 반복)
    for s_rule in slotting_rules:
        for d_rule in dispatch_rules:
            inventory = None # 처음 시작 시 재고 데이터는 None으로 초기화
            
            # 모든 시뮬레이션 일자를 순차적으로 실행
            for i, date in enumerate(unique_dates):
                # [핵심] 7일마다 창고 내부 재배치(Slotting) 갱신 로직
                # i가 0, 7, 14 ... 일 때 즉 매주의 첫날에만 새로운 재고 배치를 수행
                if i % 7 == 0:
                    # 앞으로 다가올 7일치 일자 목록 추출
                    upcoming_7 = unique_dates[i : i+7]
                    # 생산 계획 테이블에서 향후 7일간의 물량 데이터를 필터링하여 주간 계획(weekly_plan) 확보
                    weekly_plan = production_plan[production_plan['생산일자'].isin(upcoming_7)]
                    
                    # 해당 주간 계획과 선택된 슬로팅 룰(s_rule)을 기반으로 새로운 창고 배치도(inventory) 생성
                    inventory = generate_fixed_inventory(s_rule, part_list, weekly_plan, bom_df, outbound_plan, seed=100)
                
                    print(f">> {date} [{s_rule} x {d_rule}] 실행 중...")

                # 해당 날짜에 처리해야 할 당일치 출고 계획(주문서) 필터링
                date_plan = outbound_plan[outbound_plan['생산일자'] == date]
                
                # 시뮬레이터 구동: 특정 날짜, 재고 배치도, 디스패칭 룰, 당일 주문서를 인자로 넘겨 하루 동안의 로봇 작업 시뮬레이션 실행
                res = run_simulation(date, inventory, d_rule, date_plan)

                # 하루치 시뮬레이션 결과 딕셔너리에 현재 적용되었던 룰 메타데이터 추가
                res['Slotting'] = s_rule
                res['Dispatching'] = d_rule
                
                # 요약 결과 리스트에 하루치 성적 저장
                all_results_summary.append(res)

                # 이동 이력 로그에도 룰 메타데이터를 추가한 뒤 전체 로그 풀에 병합
                move_tmp = res['Move_log'].copy()
                move_tmp['Slotting'], move_tmp['Dispatching'], move_tmp['Date'] = s_rule, d_rule, date
                move_logs_all.append(move_tmp)

                # 디스패칭(배치 할당) 로그에도 동일하게 메타데이터 추가 후 병합
                disp_tmp = res['Dispatch_log'].copy()
                disp_tmp['Slotting'], disp_tmp['Dispatching'], disp_tmp['Date'] = s_rule, d_rule, date
                dispatch_logs_all.append(disp_tmp)

    # 5. 최종 분석 및 모든 조합 시각화
    # 수집된 하루 단위 요약 결과들을 하나의 묶음 데이터프레임으로 변환
    df_res = pd.DataFrame(all_results_summary)
    # 수집된 모든 로봇 이동 로그를 하나의 거대한 묶음 데이터프레임으로 통합 (인덱스 리셋)
    move_all_df = pd.concat(move_logs_all, ignore_index=True)
    
    # (1) [수정사항] 일별 텍스트 보드 대신, 주 단위 히트맵 시각화 표 (12개 조합) 출력 함수 호출
    report_weekly_scoreboard(df_res)
    
    print("\n" + "="*60)
    print(" 모든 조합(12개)에 대한 상세 리포트 그래프를 생성합니다.")
    print(" ※ 주의: 총 12개의 조합 시각화 창이 순차적으로 열립니다.")
    print("="*60)

    # (2) 12개 조합에 대한 개별 3D, 2D, Gantt 시각화 함수 순차적 호출
    for s_rule in slotting_rules:
        for d_rule in dispatch_rules:
            visualize_rule_combination(move_all_df, s_rule, d_rule)

if __name__ == "__main__":
    main()

In [ ]:
import simpy
import pandas as pd 
import numpy as np
import random
import math                         # [수정사항] 유클리드 대각선 거리를 계산하기 위해 math 모듈 추가
from itertools import combinations  # 주문 내에서 함께 출고되는 부품쌍 생성 (Kitting Slotting에 사용)
from collections import Counter     # 함께 출고되는 부품쌍의 출현 빈도 계산 (어떤 부품을 서로 가깝게 배치할지 판단하기 위함)
import matplotlib.pyplot as plt     # 시각화 및 스타일링
import seaborn as sns               # 히트맵용 시각화
import matplotlib.patches as patches # 2D 궤적 맵에서 랙(Rack)을 그리기 위함

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# --- 시스템 환경 설정 (상수 정의) ---
# 창고의 물리적 레이아웃과 물류 로봇 성능 정의
RACK_START_Y, RACK_END_Y = 4.0, 15.1       # 랙이 설치된 Y축의 시작과 끝 좌표 범위 (4.0m ~ 15.1m 사이에만 적재 가능)
AISLE_X = [1.2, 3.6, 6.0, 8.4, 10.8, 13.2] # 창고 통로(Aisle)가 위치한 X축 좌표 리스트

# 작업 스테이션 위치 (x, y, z): 레이아웃과 일치하도록 S01은 A02(3.6)와, S02는 A05(10.8)와 X좌표를 완벽히 맞춤
STATIONS = {'S01': [3.6, 0.5, 0], 'S02': [10.8, 0.5, 0]} 

CTU_SPEED_XY = 1.8  # 로봇의 XY 이동 속도 (m/s)
CTU_SPEED_Z = 0.5   # 로봇의 리프트(Z) 상하 이동 속도 (m/s)
TOTE_CAPACITY = 7   # CTU가 한 번에 운반할 수 있는 최대 토트 박스 수
SHELF_HEIGHTS = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0] # 선반의 층별 높이(Z축) 리스트 (부품의 Z좌표로 사용)

# 교차 통로(Cross Aisle) 설정
# 로봇이 다른 통로로 넘어갈 때 랙을 관통하지 않고 앞쪽으로 빠져나와서 이동하는 기준 Y 좌표 
CROSS_AISLE_Y = 3.0

# [수정] 물류기기(CTU) 운영 대수 상수화
# 하드코딩을 방지하고 향후 기획안에 명시된 "2대 or 3대 (조절 가능)" 실험을 원활하게 하기 위해 상수로 정의합니다.
NUM_CTUS = 2

# ==========================================
# 창고 레이아웃 기반 예상 소요 시간 산출 함수 (기존 거리 함수 대체)
# ==========================================
def get_estimated_travel_time(pos1, pos2): # [수정사항] 단순 거리에서 속도가 반영된 소요 시간 계산으로 변경
    """
    물류 로봇이 랙(Rack)을 뚫고 지나갈 수 없다는 물리적 제약을 반영하여
    두 위치(pos1, pos2) 사이의 '예상 이동 소요 시간'을 계산하는 함수.
    디스패칭 시 어떤 작업이 가장 '빨리' 도달할 수 있는지 판단하는 기준이 됨.
    """
    x1, y1, z1 = pos1 # 시작 지점의 x, y, z 좌표를 언패킹
    x2, y2, z2 = pos2 # 도착 지점의 x, y, z 좌표를 언패킹
    
    # 1. 같은 통로(X좌표 동일)일 경우: 단순히 Y축 깊이 차이만 계산
    if x1 == x2: # 시작점과 도착점의 X 좌표(통로)가 같은지 확인
        xy_dist = abs(y1 - y2) # 같은 통로면 Y 좌표의 차이 절대값을 평면 거리로 산출
    # 2. 다른 통로일 경우: 현재 통로에서 교차 통로(Y=3.0)로 직진하여 빠져나온 뒤, 가로(X)로 이동 후 새 통로로 진입
    else: # 다른 통로로 이동해야 하는 경우
        xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y) # 현재 Y에서 교차통로로 나가는 거리 + X축 이동 거리 + 새 통로로 들어가는 거리 합산
        
    dz = abs(z1 - z2) # Z축(높이) 차이 계산
    
    # 평면 이동 속도와 리프트 속도 중 병목이 되는(더 오래 걸리는) 진짜 소요 시간을 반환
    return max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z) 

# ==========================================
# CTU class 물류 로봇
# ==========================================
class CTU: # 물류 로봇(CTU) 객체를 정의하는 클래스
    def __init__(self, env, id, warehouse, dispatch_rule): # 로봇 초기화 메서드 (환경, ID, 소속 창고, 디스패칭 룰을 인자로 받음)
        self.env = env # SimPy 시뮬레이션 환경 객체를 로봇의 속성으로 저장
        self.id = id # CTU를 구별하기 위한 고유 번호(CTU 0 또는 1)
        self.warehouse = warehouse # 로봇이 작업할 창고 객체의 참조를 저장
        self.dispatch_rule = dispatch_rule # 현재 로봇에 적용할 작업 할당 방식(Nearest, Density 등) 저장
        
        # 초기 로봇 위치 설정: 0번 CTU는 S01에서, 1번 CTU는 S02에서 시작
        self.pos = STATIONS['S01'] if id == 0 else STATIONS['S02'] # ID가 0이면 S01 좌표, 아니면 S02 좌표를 초기 위치로 설정

        # 총 이동거리 및 작업 수 기록
        self.total_distance = 0 # 로봇이 이동한 총 누적 거리를 0으로 초기화
        self.completed_jobs = 0 # 로봇이 완료한 총 피킹 작업 수를 0으로 초기화
        self.busy = False # 로봇의 현재 상태를 대기 중(False)으로 초기화
        
        # 이 CTU가 현재 작업하기 위해 점유 중인 통로(X좌표)들을 담아두는 집합(Set) 변수 추가
        self.current_batch_aisles = set() 
        
        self.process = env.process(self.run()) # 로봇의 메인 작업 루프인 run()을 SimPy 이벤트 큐에 프로세스로 등록

    # 이동 시간 및 거리 계산 (맨해튼 직교 거리 기반)
    def calculate_travel_time(self, target_pos, is_return=False): # 복귀 여부를 알 수 있는 is_return 파라미터 기본값 False로 추가
        x1, y1, z1 = self.pos # 로봇의 현재 위치 좌표 언패킹
        x2, y2, z2 = target_pos # 로봇이 이동해야 할 목표 위치 좌표 언패킹
        
        if is_return: 
            # 스테이션으로 복귀할 경우: 
            # 랙 내부(Y>=4.0)에서 교차 통로(Y=3.0)까지는 통로를 따라 직진(Manhattan)으로 빠져나오고,
            # Y=3.0 교차 통로에서부터 스테이션(Y=0.5) 구간은 장애물이 없는 열린 공간이므로 대각선(유클리드)으로 가로지릅니다.
            dist_to_cross_aisle = abs(y1 - CROSS_AISLE_Y) 
            euclidean_dist_to_station = math.sqrt((x1 - x2)**2 + (CROSS_AISLE_Y - y2)**2)
            xy_dist = dist_to_cross_aisle + euclidean_dist_to_station
        else:
            if x1 == x2: # 출발지와 목적지의 통로(X좌표)가 동일한지 확인
                xy_dist = abs(y1 - y2) # 통로가 같으면 랙 안쪽으로 직진하는 거리만 계산
            else: # 통로가 다른 경우
                xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y) # 앞쪽 교차 통로(Y=3.0)로 우회하는 평면 궤적 거리 계산
            
        dz = abs(z1 - z2) # Z축(상하 리프트) 이동 거리 계산
        
        travel_time = max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z) # 평면 주행과 리프트 승강은 동시 수행되므로 둘 중 더 오래 걸리는 시간을 소요 시간으로 산정
        travel_distance = xy_dist + dz # 물리적인 총 모터 이동 거리(주행 거리 + 리프트 이동 거리) 합산
        
        return travel_time, travel_distance # 계산된 이동 시간과 총 이동 거리를 튜플로 반환

    # CTU의 실제 작업 수행 프로세스
    def run(self): # 시간 흐름에 따라 CTU가 지속적으로 수행할 제너레이터 메서드
        while True: # 시뮬레이션 종료 시까지 무한 반복
            if self.warehouse.job_queue: # 창고에 처리 대기 중인 주문 작업이 남아있는지 확인
                # 무조건 busy를 True로 바꾸지 않고, 겹치지 않는 배치를 받아올 수 일 때만 상태를 변경하도록 로직 순서 변경
                batch = self.warehouse.apply_dispatching(self, self.dispatch_rule) 
                
                if batch: # 겹치지 않는 할당받은 작업 묶음이 존재한다면
                    self.busy = True # 비로소 작업 상태(True)로 전환
                    
                    # 할당받은 작업 배치에 포함된 모든 통로의 X좌표를 추출하여 현재 CTU의 점유 통로로 등록
                    # (이후 다른 CTU가 디스패칭을 수행할 때 이 통로의 작업들은 투명 인간처럼 무시됨)
                    self.current_batch_aisles = set(job['pos'][0] for job in batch)
                    
                    # [수정사항: 1번 이슈 해결] 거리 기준 정렬에서 '예상 소요 시간' 기준 정렬로 변경
                    batch.sort(key=lambda j: get_estimated_travel_time(self.pos, j['pos'])) 
                    
                    for job in batch: # 할당받은 각 작업(부품 피킹)을 순회
                        t, d = self.calculate_travel_time(job['pos']) # 해당 부품 위치까지의 이동 시간(t)과 거리(d)를 계산
                        
                        self.warehouse.move_log.append({ # 분석을 위해 로봇의 이동 상세 이력을 창고의 로그 리스트에 추가
                            'time': self.env.now, # 현재 시뮬레이션 시간 기록
                            'ctu_id': self.id, # 어떤 로봇이 이동했는지 ID 기록
                            'from_x': self.pos[0], 'from_y': self.pos[1], 'from_z': self.pos[2], # 출발 지점 X, Y, Z 좌표 기록
                            'to_x': job['pos'][0], 'to_y': job['pos'][1], 'to_z': job['pos'][2], # 도착(피킹) 지점 X, Y, Z 좌표 기록
                            'item_id': job['item_id'], # 피킹한 대상 부품 코드 기록
                            'travel_time': t, 'travel_distance': d, # 해당 구간 이동 소요 시간과 이동 거리 기록
                            'dispatch_rule': self.dispatch_rule, # 당시 적용된 디스패칭 룰 기록
                        })
                        
                        yield self.env.timeout(t) # 계산된 이동 시간(t)만큼 시뮬레이션 시간을 흘려보냄 
                        
                        self.pos = job['pos'] # 로봇의 현재 위치를 방금 피킹을 완료한 목적지 좌표로 갱신
                        self.total_distance += d # 로봇의 누적 이동 거리에 방금 이동한 거리를 더함
                        self.completed_jobs += 1 # 완료한 피킹 작업 개수 1 증가
                        
                    # 마지막 피킹 위치의 X좌표에 따라 복귀할 지정 스테이션 결정
                    last_job_x = self.pos[0] # 배치의 모든 피킹이 끝난 후 로봇의 현재(마지막) 통로 X 좌표 확인
                    target_station = STATIONS['S01'] if last_job_x <= 6.0 else STATIONS['S02'] # 마지막 X 좌표가 6.0 이하(A01~A03)면 S01, 초과면 S02를 복귀 지점으로 결정
                    
                    # 복귀할 때는 is_return=True를 넘겨주어 Y=3.0 앞쪽에서는 유클리드 거리로 대각선 복귀하도록 호출
                    t, d = self.calculate_travel_time(target_station, is_return=True) 
                    
                    self.warehouse.move_log.append({ # 분석을 위해 로봇의 스테이션 복귀 이력을 창고 로그에 추가
                        'time': self.env.now, # 현재 시뮬레이션 시간 기록
                        'ctu_id': self.id, # 복귀하는 로봇 ID 기록
                        'from_x': self.pos[0], 'from_y': self.pos[1], 'from_z': self.pos[2], # 마지막 피킹 지점 좌표 기록
                        'to_x': target_station[0], 'to_y': target_station[1], 'to_z': target_station[2], # 목적지 스테이션 좌표 기록
                        'item_id': 'RETURN', # 피킹 작업이 아닌 스테이션 복귀임을 의미하는 태그 삽입
                        'travel_time': t, 'travel_distance': d, # 복귀 구간 소요 시간 및 거리 기록
                        'dispatch_rule': self.dispatch_rule, # 당시 적용된 디스패칭 룰 기록
                    })
                    
                    yield self.env.timeout(t) # 복귀 시간만큼 시뮬레이션 시간 지연
                    self.pos = target_station # 복귀가 완료되었으므로 로봇 위치를 스테이션 좌표로 갱신
                    self.total_distance += d # 복귀 이동 거리 누적 더하기
                
                    # 모든 작업과 복귀를 마쳤으므로 점유했던 통로들을 깨끗하게 비워 해제
                    self.current_batch_aisles.clear()
                    self.busy = False # 로봇 상태를 다시 대기 중(False)으로 변경
                else: 
                    # 큐에 짐은 남아있으나, 다른 CTU가 모두 점유하고 있어 배치를 받지 못한 경우 1초간 대기하며 눈치보기
                    yield self.env.timeout(1) 
            else: # 아예 창고 큐 전체에 남은 할당 작업이 없는 경우
                yield self.env.timeout(1) # 무한 루프 과부하를 막기 위해 1초 대기 후 다시 작업 확인


# ==========================================
# 창고 시스템 클래스 정의
# ==========================================
class Warehouse: # 전체 창고, 재고, 큐, 로그를 통합 관리하는 클래스
    def __init__(self, env, inventory, plan_df): # 창고 초기화 메서드
        self.env = env # SimPy 시뮬레이션 환경 저장
        self.inventory = inventory # 슬로팅 룰에 의해 생성된 {부품: 위치} 맵핑 데이터 저장
        self.plan_df = plan_df # 금일 처리해야 할 전체 출고 계획 주문 데이터 저장
        self.job_queue = [] # 실시간으로 쏟아지는 주문(부품 피킹)들이 대기하는 리스트(큐) 초기화
        
        self.dispatch_log = [] # 디스패칭 룰에 의해 배치 작업이 어떻게 생성되었는지 기록할 리스트
        self.move_log = [] # 로봇의 점대점 이동 상세 이력을 기록할 리스트
        
        self.ctus = [] # 교착(Deadlock) 방지를 위해 창고가 전체 CTU들의 객체 정보를 알고 있도록 리스트 초기화

    def apply_dispatching(self, ctu, rule): # 룰에 따라 큐에서 작업을 꺼내 로봇에 할당해주는 메서드
        if not self.job_queue: return [] # 만약 큐가 비어있다면 빈 리스트 반환하여 작업 할당 취소

        # 병목 현상 방지: 현재 작업을 호출한 본인이 아닌 다른 CTU가 작업 중인 통로(X좌표) 목록을 수집
        occupied_aisles = set()
        for c in self.ctus:
            if c.id != ctu.id and c.busy: # 자신이 아니며, 현재 작업을 수행 중인 CTU인 경우
                occupied_aisles.update(c.current_batch_aisles) # 해당 CTU가 점유 중인 통로 목록을 블랙리스트(occupied_aisles)에 병합

        # 전체 대기 큐의 작업들 중에서 점유(블랙리스트)되지 않은 통로에 위치한 작업들만 골라 가용 풀을 생성
        available_jobs = [j for j in self.job_queue if j['pos'][0] not in occupied_aisles]

        # 만약 겹치지 않는 작업이 단 하나도 없다면 당장 할당을 포기하고 빈 리스트를 반환하여 안전하게 대기 유도
        if not available_jobs:
            return [] 

        # 1. Nearest
        if rule == 'nearest': # 디스패칭 룰이 가장 가까운 작업 우선(Nearest)인 경우
            # 소요 시간(Time) 기준으로 가장 빨리 도착할 수 있는 작업 정렬
            available_jobs.sort(key=lambda j: get_estimated_travel_time(ctu.pos, j['pos'])) 
            batch = [available_jobs.pop(0)] # 정렬된 가용 풀에서 제일 앞(가장 가까운) 작업을 Seed(기준 작업)로 뽑아 배치에 첫 번째로 담음
            
            if available_jobs: # 가용 풀에 남은 작업이 더 있다면
                available_jobs.sort(key=lambda j: get_estimated_travel_time(batch[0]['pos'], j['pos'])) # 방금 뽑은 Seed 작업 위치를 기준으로 가까운 순으로 재정렬
                extra_count = min(len(available_jobs), TOTE_CAPACITY - 1) # 남은 가용 작업 수와 로봇 여유 적재량(7-1=6) 중 작은 값을 추가 작업 수로 확정
                batch.extend(available_jobs[:extra_count]) # Seed 주변의 추가 가용 작업들을 배치 리스트에 병합
                
            # 원본 job_queue에서 최종 할당된 배치의 작업들만 찾아내어 안전하게 삭제
            for j in batch:
                self.job_queue.remove(j)
                
            self.dispatch_log.append({ # 배치 구성 완료 후 배치 로그 기록
                'time': ctu.env.now, 'ctu_id': ctu.id, 'rule': 'nearest', # 시간, 로봇, 적용 룰 기록
                'seed_item': batch[0]['item_id'], 'batch_items': [j['item_id'] for j in batch], # 기준 부품과 전체 배치 부품 목록 기록
                'batch_size': len(batch), # 이 배치의 총 작업 수 기록
            })
            return batch # 완성된 묶음(배치) 반환

        # 2. Density
        elif rule == 'density': # 디스패칭 룰이 작업 밀집도 우선(Density)인 경우
            K = min(10, len(available_jobs)) # 가용 작업 중 로봇에서 가장 가까운 상위 10개만 Seed 후보로 추림
            candidate_seeds = sorted(available_jobs, key=lambda j: get_estimated_travel_time(ctu.pos, j['pos']))[:K] 
            
            best_seed = None # 최적의 Seed 작업을 저장할 변수 초기화
            max_density = -1 # 가장 높은 밀집도 점수를 저장할 변수 초기화
            r_y = 2.65 # 같은 통로 내에서 밀집도를 계산할 검색 반경(2.65m) 설정
            
            for seed in candidate_seeds: # 후보 Seed 10개 각각에 대해 주변 작업 밀집도 조사
                density = 0 # 현재 평가 중인 Seed의 밀집도 카운터 0으로 시작
                for job in available_jobs: # 밀집도를 계산할 때도 다른 로봇이 점유한 통로의 작업은 철저히 배제
                    if job['pos'][0] == seed['pos'][0]: # 평가 대상 작업이 Seed 작업과 같은 통로(X좌표)에 있는지 확인
                        dy = abs(job['pos'][1] - seed['pos'][1]) # 같은 통로라면 Y좌표(깊이) 차이 계산
                        if dy <= r_y: # Y축 거리가 반경(2.65m) 이내라면
                            density += 1 # 주변 작업이므로 밀집도 카운트 1 증가
                
                if density > max_density: # 현재 계산한 밀집도가 역대 최대 밀집도보다 높다면
                    max_density = density # 최대 밀집도 갱신
                    best_seed = seed # 최적 Seed를 현재 Seed로 변경
                    
            available_jobs.sort(key=lambda j: get_estimated_travel_time(best_seed['pos'], j['pos'])) # 찾아낸 최적 밀집 Seed를 기준으로 가용 풀을 정렬
            
            batch_size = min(len(available_jobs), TOTE_CAPACITY) # 가용 남은 물량과 로봇 최대 용량 중 묶을 크기 결정
            best_batch = available_jobs[:batch_size] # 정렬된 가용 큐에서 위에서 결정한 개수만큼 묶음 생성
            
            # 원본 전체 job_queue에서 배치로 뽑혀나간 작업들을 삭제
            for j in best_batch:
                self.job_queue.remove(j)
            
            self.dispatch_log.append({ # 밀집도 기반 배치 생성 로그 기록
                'time': ctu.env.now, 'ctu_id': ctu.id, 'rule': 'density', # 시간, 로봇 ID, 룰 종류 기록
                'seed_item': best_seed['item_id'], 'batch_items': [j['item_id'] for j in best_batch], # 기준 부품 및 배치 내 부품 리스트 기록
                'batch_size': len(best_batch), 'local_density': max_density, # 배치 크기 및 해당 지역의 계산된 밀집도 점수 기록
            })
            return best_batch # 생성된 밀집 배치 반환

        # 3. Load-balance
        elif rule == 'load_balance': # 디스패칭 룰이 최단 소요 시간 예측(Load-balance)인 경우
            K = min(10, len(available_jobs)) # 가용 풀 작업 중 가까운 상위 10개를 Seed 후보로 추출
            nearest_jobs = sorted(available_jobs, key=lambda j: get_estimated_travel_time(ctu.pos, j['pos'])) 
            candidate_seeds = nearest_jobs[:K] 
            
            best_batch = [] # 최단 시간으로 처리될 최적의 배치를 저장할 리스트 초기화
            min_eta = float('inf') # 최소 예상 소요 시간(ETA)을 무한대로 초기화 (비교 갱신용)
            best_seed_item = None # 최적 배치에 사용된 Seed 부품 코드를 저장할 변수 초기화
            
            for seed in candidate_seeds: # 10개의 각 후보 Seed를 가상으로 배치로 묶어봄
                # 임시 큐를 가용 작업(available_jobs)만을 대상으로 Seed 기준 가까운 순 정렬
                temp_queue = sorted(available_jobs, key=lambda j: get_estimated_travel_time(seed['pos'], j['pos'])) 
                temp_batch = temp_queue[:TOTE_CAPACITY] # 임시 가용 큐에서 최대 용량만큼 가상 묶음 구성
                
                eta = 0 # 이 가상 배치를 처리할 때의 예상 소요 시간(ETA) 0으로 초기화
                current_pos = ctu.pos  # 로봇의 이동을 시뮬레이션하기 위해 가상 위치를 현재 로봇 위치로 설정
                
                for job in temp_batch: # 가상 배치의 작업들을 순서대로 가상 방문
                    x1, y1, z1 = current_pos # 가상 현재 좌표 언패킹
                    x2, y2, z2 = job['pos'] # 가상 이동 목표 좌표 언패킹
                    
                    if x1 == x2: # 통로가 같은지 확인
                        xy_dist = abs(y1 - y2) # 평면 직진 거리 계산
                    else: # 통로가 다르면
                        xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y) # 교차 통로 우회 거리 계산
                        
                    dz = abs(z1 - z2) # 리프트 이동 거리 계산
                    time_to_job = max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z) # 현재 지점에서 목표 작업까지의 시간 계산
                    eta += time_to_job # 계산된 구간 시간을 가상 총 소요 시간(ETA)에 누적
                    current_pos = job['pos']  # 가상 로봇 위치를 방금 계산한 작업 지점으로 갱신
                
                # --- 스테이션 복귀 시간 ETA에 반영 ---
                last_x, last_y, last_z = current_pos
                target_station = STATIONS['S01'] if last_x <= 6.0 else STATIONS['S02']
                
                # 복귀 경로 이동 거리 산출 (is_return=True 방식 적용)
                dist_to_cross_aisle = abs(last_y - CROSS_AISLE_Y)
                euclidean_dist_to_station = math.sqrt((last_x - target_station[0])**2 + (CROSS_AISLE_Y - target_station[1])**2)
                return_xy_dist = dist_to_cross_aisle + euclidean_dist_to_station
                return_dz = abs(last_z - target_station[2])
                
                return_time = max(return_xy_dist / CTU_SPEED_XY, return_dz / CTU_SPEED_Z)
                eta += return_time # 총 소요 시간에 복귀 시간을 합산해야 진짜 Makespan 산출이 가능
                # -----------------------------------------------------------------
                
                if eta < min_eta: # 이 가상 배치의 총 예상 시간이 이전 최소값보다 더 짧다면
                    min_eta = eta # 최소 예상 시간 갱신
                    best_batch = temp_batch # 최적의 배치를 방금 만든 가상 배치로 교체
                    best_seed_item = seed['item_id'] # 그때 사용된 Seed 부품명 저장
            
            # 최종 확정된 배치의 부품들을 진짜 대기 큐(job_queue)에서 안전하게 지우기
            for job in best_batch: 
                self.job_queue.remove(job) 
                
            self.dispatch_log.append({ # 로드 밸런스 기반 배치 생성 로그 기록
                'time': ctu.env.now, 'ctu_id': ctu.id, 'rule': 'load_balance', # 기본 메타데이터 기록
                'seed_item': best_seed_item, 'batch_items': [j['item_id'] for j in best_batch], # Seed 정보와 할당 부품 리스트 기록
                'batch_size': len(best_batch), 'eta': min_eta, # 배치 크기와 시뮬레이션 전 예측했던 예상 종료 시간(ETA) 기록
            })
            return best_batch # 가장 빨리 끝나는 최적 배치 반환
        
        # [수정] 4. RL (강화학습 기반 메타 디스패처)
        elif rule == 'rl':
            # 향후 제안 모델인 RL 에이전트를 연결하기 위한 뼈대(Stub) 부분입니다.
            # 이곳에 창고의 현재 상태(State)를 추출하여 보상(Reward)을 최적화하는 
            # 3가지 행동(Nearest, Density, Load-balance) 중 하나를 예측하는 모델 추론 코드가 추가될 예정입니다.
            pass


#슬로팅 로직 수정 
def generate_fixed_inventory(rule_type, part_list, weekly_prod_plan, bom_df, outbound_plan, seed=1234):
    """
    교수님 피드백 반영: 
    1. 제품별 생산 계획(물량)을 고려
    2. kitting 시 같은 제품의 부품들을 물리적으로 인접하게 배치
    """
    random.seed(seed)
    inventory = {}
    
    # 1. 제품별 주간 생산 물량 계산
    prod_vol = weekly_prod_plan.groupby('제품코드').size().to_dict()
    # 물량이 많은 순서대로 제품 정렬
    sorted_products = sorted(prod_vol.keys(), key=lambda x: prod_vol.get(x, 0), reverse=True)

    # 2. 가용 슬롯 생성 및 정렬 (스테이션에서 가까운 순)
    # [수정] PPT 기획안 기준 완벽 동기화: 랙 1개 길이에 16개의 박스(Bin) 적재
    # 통로(Aisle)를 따라 로봇이 정차하는 Y축 위치를 현실의 박스 규격(600mm)에 맞춰 정확히 16등분합니다.
    y_positions = np.linspace(RACK_START_Y, RACK_END_Y, num=16)
    candidate_slots = []
    for x in AISLE_X:
        for y in y_positions:
            for z in SHELF_HEIGHTS:
                # [수정] 좌/우 양면 랙(Back-to-Back) 동시 커버 로직
                # 로봇이 하나의 통로 X에 진입하여 특정 Y, Z 좌표에 멈추면,
                # 좌측 랙과 우측 랙 양쪽의 Bin을 모두 사용할 수 있으므로 2개의 가용 슬롯을 생성합니다.
                # 총 슬롯 수 = 6개 통로 * 16개 Y위치 * 7층 * 2(좌우) = 기획안과 동일한 정확히 1,344 Bin 확보
                candidate_slots.append((x, round(y,2), z)) # 로봇 기준 왼쪽 랙(Left Rack)의 가용 Bin 공간
                candidate_slots.append((x, round(y,2), z)) # 로봇 기준 오른쪽 랙(Right Rack)의 가용 Bin 공간

    def est_time_to_station(pos): # 각 슬롯에서 스테이션까지의 예상 접근 시간을 구하는 내부 함수
        x, y, z = pos
        min_time = float('inf')
        for st in STATIONS.values():
            dist = abs(y - CROSS_AISLE_Y) + math.sqrt((x - st[0])**2 + (CROSS_AISLE_Y - st[1])**2)
            t = max(dist / CTU_SPEED_XY, abs(z - st[2]) / CTU_SPEED_Z)
            if t < min_time: min_time = t
        return min_time

    candidate_slots.sort(key=est_time_to_station) # 슬롯들을 스테이션 도달 시간이 짧은(가까운) 순으로 미리 정렬

    # 3. 전략별 부품 배치 순서 결정 (Parts Ordering)
    parts_order = []
    used_parts = set()

    # generate_fixed_inventory 함수 내부의 Kitting 전략 부분 수정 제안
    # --- generate_fixed_inventory 함수 내부의 kitting 조건절 수정 ---
    if rule_type == "kitting":
        # 1. 함께 출고되는 부품 쌍(Pair)의 빈도 계산
        pair_counter = Counter()
        for _, group in outbound_plan.groupby('No.'):
            parts = sorted(group['부품코드'].unique())
            if len(parts) > 1:
                for pair in combinations(parts, 2):
                    pair_counter[pair] += 1
        
        # 2. 빈도가 높은 쌍부터 확인하며 '묶음(Cluster)' 생성
        # 조건: 한 묶음은 최소 2개(쌍으로 시작), 최대 8개까지
        clusters = []
        assigned_parts = set()
        
        # 빈도순으로 정렬된 부품 쌍들
        sorted_pairs = [pair for pair, count in pair_counter.most_common()]
        
        for p1, p2 in sorted_pairs:
            # 케이스 1: 둘 다 아직 어느 묶음에도 속하지 않음 -> 새로운 묶음 생성 (최소 2개 만족)
            if p1 not in assigned_parts and p2 not in assigned_parts:
                clusters.append([p1, p2])
                assigned_parts.add(p1)
                assigned_parts.add(p2)
            
            # 케이스 2: p1은 이미 묶음에 있고 p2는 없음 -> p1이 있는 묶음에 p2 추가 (최대 8개 체크)
            elif p1 in assigned_parts and p2 not in assigned_parts:
                for cluster in clusters:
                    if p1 in cluster and len(cluster) < 8:
                        cluster.append(p2)
                        assigned_parts.add(p2)
                        break
            
            # 케이스 3: p2는 이미 묶음에 있고 p1은 없음 -> p2가 있는 묶음에 p1 추가 (최대 8개 체크)
            elif p2 in assigned_parts and p1 not in assigned_parts:
                for cluster in clusters:
                    if p2 in cluster and len(cluster) < 8:
                        cluster.append(p1)
                        assigned_parts.add(p1)
                        break
        
        # 3. 생성된 묶음들을 순서대로 parts_order에 추가
        # (묶음 내의 부품들은 parts_order에서 나란히 배치되므로, 결과적으로 창고에서도 가깝게 배치됨)
        parts_order = []
        for cluster in clusters:
            parts_order.extend(cluster)
            
        # 4. 묶이지 않은 나머지 부품들(단독 출고 등)을 뒤에 추가
        remaining_parts = [p for p in part_list if p not in assigned_parts]
        parts_order.extend(remaining_parts)

    elif rule_type == "characteristic":
        # 생산 물량이 가장 많은 Top 제품의 부품들을 전진 배치
        if sorted_products:
            top_p = sorted_products[0]
            top_parts = set(bom_df[bom_df['product_code'] == top_p]['part_code'].unique())
            parts_order = [p for p in part_list if p in top_parts] + \
                          [p for p in part_list if p not in top_parts]
        else:
            parts_order = list(part_list)

    elif rule_type == "abc":
        # freq_time과 유사하게 정렬 후 등급 개념 부여 (여기서는 정렬만 수행)
        # 각 부품이 포함된 제품들의 전체 생산 물량(vol)을 합산하여 부품의 총 수요(part_demand)를 계산합니다.
        part_demand = {}
        for p_code, vol in prod_vol.items(): # 모든 제품의 주간 생산 물량을 순회
            p_parts = bom_df[bom_df['product_code'] == p_code]['part_code'].unique() # 해당 제품에 들어가는 부품 목록 추출
            for p in p_parts:
                part_demand[p] = part_demand.get(p, 0) + vol # 각 부품에 생산 물량 가중치를 누적
        # 수요(생산 물량 합계)가 높은 부품일수록 리스트의 앞쪽에 오도록 내림차순(reverse=True) 정렬합니다.
        parts_order = sorted(part_list, key=lambda x: part_demand.get(x, 0), reverse=True)

    else: # random
        # 슬로팅 규칙이 무작위(Random)인 경우
        parts_order = list(part_list) # 전체 부품 리스트를 복사한 뒤
        random.shuffle(parts_order)   # 부품의 순서를 완전히 무작위로 섞습니다. (나중에 슬롯에 차례대로 배정되므로 무작위 배치가 됨)

    # 4. 부품과 슬롯 매핑 (1:1)
    for part, slot in zip(parts_order, candidate_slots):
        inventory[part] = list(slot)

    return inventory

# ==========================================
# 하루 단위 시뮬레이션 실행
# ==========================================
def run_simulation(date, inventory, dispatch_rule, date_plan): # 특정 일자의 작업을 시뮬레이션하는 메인 구동 함수
    env = simpy.Environment() # 이산 사건 시뮬레이션의 근간이 되는 SimPy 환경 객체 생성
    warehouse = Warehouse(env, inventory, date_plan) # 위에서 만든 재고와 일일 계획을 바탕으로 창고 객체 인스턴스화
    
    # [수정] 하드코딩된 대수(range(2)) 대신 상수(NUM_CTUS)를 사용하여 2대, 3대 조절 실험 유연성 확보
    ctus = [CTU(env, i, warehouse, dispatch_rule) for i in range(NUM_CTUS)] # 창고 내에서 주어진 디스패칭 룰에 따라 지정된 대수의 CTU 객체 생성
    warehouse.ctus = ctus # 창고 시스템이 전체 로봇의 위치(점유 통로)를 알 수 있도록 방금 생성한 CTU 리스트를 창고 속성에 주입
    
    def job_gen(env, wh): # 계획에 맞춰 주문 발생을 모사하는 내부 제너레이터 함수
        # 일관된 결과를 위해 각 날짜마다 시드를 일정하게 맞추어 주문 발생 타이밍 통제
        random.seed(hash(date) % 10000) 
        
        for _, group in wh.plan_df.groupby('No.'): # 일일 주문 계획을 주문번호(No.) 별로 묶어 순회
            yield env.timeout(random.uniform(2, 5))  # 각 주문이 시스템에 인입되는 데 걸리는 무작위 시간(2~5초) 간격 부여
            for _, row in group.iterrows(): # 하나의 주문 묶음 내 포함된 부품 리스트 순회
                p_id = row['부품코드'] # 작업해야 할 부품 코드 추출
                if p_id in wh.inventory: # 재고 맵에 이 부품의 위치가 존재하는지 안전 검사
                    wh.job_queue.append({'item_id': p_id, 'pos': wh.inventory[p_id], 'time': env.now}) # 창고 대기 큐에 이 부품의 위치, 주문 발생 시간 기록

    def monitor(env, warehouse, gen_proc, ctus): # 시뮬레이션 종료 시점을 감시하는 내부 함수
        yield gen_proc # 주문 발생 제너레이터(job_gen)가 모든 주문 인입을 끝낼 때까지 대기
        while warehouse.job_queue or any(c.busy for c in ctus): # 주문 생성이 끝난 후에도 창고 큐에 작업이 남아있거나, 아직 작업 중인 로봇이 있다면
            yield env.timeout(1)  # 1초씩 시간을 흐르게 하며 모니터링 반복 루프 지속 (모두 종료되면 루프 탈출)

    gen_proc = env.process(job_gen(env, warehouse)) # 주문 발생 제너레이터를 SimPy 프로세스로 등록
    env.run(until=env.process(monitor(env, warehouse, gen_proc, ctus))) # 모니터 함수(프로세스)가 종료될 때까지 전체 시뮬레이션 엔진 가동
    
    return { # 하루치 시뮬레이션이 끝난 뒤 분석용 데이터 반환
        'Date': date,  # 시뮬레이션 실행 날짜
        'Time': round(env.now, 1),  # 시뮬레이션의 마지막 순간(총 작업 소요 시간) 반환
        'Distance': round(sum(c.total_distance for c in ctus), 1), # 두 로봇의 이동 거리 총합산 반환
        'Slotting': None,      # 메인에서 채워넣을 예정
        'Dispatching': None,   # 메인에서 채워넣을 예정
        'Move_log': pd.DataFrame(warehouse.move_log), # 쌓인 로봇 이동 이력 로그를 데이터프레임으로 변환하여 반환
        'Dispatch_log': pd.DataFrame(warehouse.dispatch_log), # 쌓인 배치 할당 로그 데이터프레임 변환 반환
    }

# ==========================================
# 개별 알고리즘 조합 통합 시각화 
# ==========================================
def visualize_rule_combination(move_all_df, slot_rule, disp_rule): # 호출 시마다 특정 슬로팅/디스패칭 조합을 시각화하도록 매개변수 추가
    """
    입력받은 슬로팅(slot_rule)과 디스패칭(disp_rule) 조합의 시뮬레이션 결과를 
    3가지 방식(3D Scatter, 2D Trajectory, Gantt Chart)으로 시각화합니다.
    (기존 Kitting + Load_balance 전용 함수를 모든 조합에 범용적으로 쓸 수 있게 확장)
    """
    # 1. 데이터 필터링: 입력받은 조합(slot_rule, disp_rule) 추출
    target_df = move_all_df[(move_all_df['Slotting'] == slot_rule) &  # 전체 이력에서 슬로팅 룰 조건 필터링
                            (move_all_df['Dispatching'] == disp_rule)].copy() # 디스패칭 룰 조건까지 AND 결합하여 복사
    
    if target_df.empty: # 해당 조합의 데이터가 비어있다면
        print(f"해당 조건({slot_rule} + {disp_rule})의 데이터가 없습니다.") # 에러 메시지 출력 후 스킵
        return # 함수 종료
        
    # 시각화가 너무 복잡해지는 것을 막기 위해 첫 번째 시뮬레이션 날짜만 선택
    sample_date = target_df['Date'].iloc[0] # 타겟 데이터프레임에서 가장 첫 번째 줄의 시뮬레이션 날짜 문자열 추출
    df_day = target_df[target_df['Date'] == sample_date].copy() # 전체 데이터 중 방금 추출한 단일 날짜의 결과만 필터링하여 복사
    
    print(f"\n=== [시각화 진행 중: Date {sample_date} | {slot_rule} × {disp_rule}] ===") # 현재 시각화하는 조합과 날짜 정보를 터미널에 명확히 안내

    # ---------------------------------------------------------
    # [1] 3D Scatter Plot: 부품 피킹 위치 분포
    # ---------------------------------------------------------
    fig1 = plt.figure(figsize=(10, 8)) # 3D 산점도를 그리기 위해 가로 10, 세로 8 크기의 새 피규어 생성
    ax1 = fig1.add_subplot(111, projection='3d') # 위 피규어의 첫 번째 영역에 3D 투영(projection)을 활성화한 축 객체 생성
    picks = df_day[df_day['item_id'] != 'RETURN'] # 스테이션 복귀는 제외하고 실제 부품 좌표 데이터만 추출
    pick_counts = picks.groupby(['to_x', 'to_y', 'to_z', 'item_id']).size().reset_index(name='count') # 동일한 X,Y,Z 위치에 피킹이 몇 번 일어났는지 집계하여 count 열로 명명
    
    scatter = ax1.scatter(pick_counts['to_x'], pick_counts['to_y'], pick_counts['to_z'],  # 추출한 x, y, z 좌표 리스트를 기반으로 3D 산점도 정밀하게 그리기
                          s=pick_counts['count']*50, c=pick_counts['to_z'], cmap='viridis', alpha=0.7) # 빈도(count)가 클수록 점 크기(s) 확대, 층수(Z)에 따라 색상 분리, 반투명도 지정
    
    ax1.set_title(f'① 3D Scatter: 피킹 위치 분포 ({slot_rule} × {disp_rule})', fontsize=14, fontweight='bold') # 3D 플롯 제목에 현재 적용 중인 룰 변수를 동적 삽입
    ax1.set_xlabel('통로 X (Aisle)') # 3D 플롯 X축 레이블 지정
    ax1.set_ylabel('랙 깊이 Y') # 3D 플롯 Y축 레이블 지정
    ax1.set_zlabel('선반 높이 Z') # 3D 플롯 높이 Z축 레이블 지정
    fig1.colorbar(scatter, ax=ax1, label='Z축 높이 (층수)') # Z 높이에 따른 범례 역할을 하는 컬러바를 우측에 부착
    plt.tight_layout() # 요소 간 겹침 방지 여백 조정
    plt.show() # 3D 산점도 화면 출력

    # ---------------------------------------------------------
    # [2] 2D Trajectory Map: 로봇 이동 궤적 밀집도 (Heatmap 스타일 렌더링)
    # ---------------------------------------------------------
    fig2, ax2 = plt.subplots(figsize=(12, 8)) # 로봇 궤적을 그릴 12x8 사이즈 피규어와 2D 축 생성
    
    # 랙 및 스테이션 배경 그리기: 기획안 도면(R01 ~ R12)과 완전히 동일한 X좌표와 폭(Width)으로 백투백(Back-to-Back) 랙 덩어리들을 생성
    rack_spans = [(0, 0.6), (1.8, 1.2), (4.2, 1.2), (6.6, 1.2), (9.0, 1.2), (11.4, 1.2), (13.8, 0.6)] # 구조: (단일 0.6m) - (양면 1.2m) * 5 - (단일 0.6m), 통로 간격은 1.2m씩
    for start_x, width in rack_spans: # 랙 덩어리 좌표 배열을 순회
        ax2.add_patch(patches.Rectangle((start_x, RACK_START_Y), width, RACK_END_Y - RACK_START_Y, color='lightgray')) # 도면에 맞게 회색 랙 덩어리 배치
    
    for x in AISLE_X: # 통로 라벨 추가
        ax2.text(x, RACK_END_Y + 0.5, f"Aisle {x}", ha='center', fontsize=9) # 각 통로 위에 번호 텍스트 부착
        
    ax2.axhline(CROSS_AISLE_Y, color='black', linestyle='--', alpha=0.3, label='교차 통로 (Cross Aisle)') # 로봇이 교차하여 지나가는 Y축(2.0) 위치에 검은 점선 가이드라인 표시
    
    ax2.add_patch(patches.Rectangle((2.6, 0), 2.0, 1.0, color='lightgreen', label='Station (S01, S02)')) # 작업 스테이션 S01 위치에 연녹색 박스 도형 생성 및 배치
    ax2.add_patch(patches.Rectangle((9.8, 0), 2.0, 1.0, color='lightgreen')) # 작업 스테이션 S02 위치에 도형 배치
    ax2.text(3.6, 0.5, 'S01', ha='center', va='center', fontweight='bold') # S01 박스 정중앙에 'S01' 텍스트 레이블 추가
    ax2.text(10.8, 0.5, 'S02', ha='center', va='center', fontweight='bold') # S02 박스 내부 텍스트 레이블 추가

    # ----- 핵심 로직: 궤적을 5cm 단위의 점(Point)으로 쪼개 통행량 카운트 -----
    traffic_counts = {} # 특정 (x, y) 좌표를 지날 때마다 누적 횟수를 저장할 빈 딕셔너리 생성

    def add_line_segment(sx, sy, ex, ey): # 시뮬레이션의 선형 궤적을 점 단위로 분해하는 내부 함수
        """시작점과 끝점을 받아 그 사이의 경로를 5cm 간격의 점으로 변환 후 카운트 증가"""
        # [수정사항] 궤적 렌더링 시에도 대각선 이동을 매끄럽게 그릴 수 있도록 맨해튼 방식이 아닌 유클리드 거리로 점의 개수를 산출
        dist = math.sqrt((sx - ex)**2 + (sy - ey)**2) 
        steps = int(dist / 0.05) + 1 # 0.05m(5cm) 해상도로 경로를 쪼개어 생성할 점의 전체 개수 계산
        for i in range(steps): # 계산된 점의 개수만큼 루프 반복
            px = sx + (ex - sx) * i / max(1, steps - 1) # i번째 점의 X 좌표 보간 계산
            py = sy + (ey - sy) * i / max(1, steps - 1) # i번째 점의 Y 좌표 보간 계산
            coord = (round(px, 2), round(py, 2)) # 계산된 점을 소수점 2자리로 정규화하여 튜플 생성
            traffic_counts[coord] = traffic_counts.get(coord, 0) + 1 # 딕셔너리에 해당 좌표 키가 있으면 값을 1 증가, 없으면 1로 초기화 (통행 빈도 누적)

    # 로봇의 모든 점대점 이동 이력 순회
    for _, row in df_day.iterrows(): # 해당 날짜 로봇의 모든 이동 이력을 한 줄씩 순회
        x1, y1 = row['from_x'], row['from_y'] # 출발점 X, Y 좌표 추출
        x2, y2 = row['to_x'], row['to_y'] # 도착점 X, Y 좌표 추출
        
        if row['item_id'] == 'RETURN': 
            # [수정사항] 복귀 궤적: 랙에서 빠져나와 교차통로(Y=3.0)까지는 직진 이동 후, 스테이션까지 유클리드(대각선) 궤적을 추가
            add_line_segment(x1, y1, x1, CROSS_AISLE_Y) # 랙에서 교차 통로로 나오기
            add_line_segment(x1, CROSS_AISLE_Y, x2, y2) # 교차 통로에서 스테이션으로 대각선 횡단
        else:
            if x1 == x2: # 출발 통로와 도착 통로가 같아 꺾을 필요가 없다면 (한 번에 직진)
                add_line_segment(x1, y1, x2, y2) # 해당 직진 구간 전체를 점으로 쪼개 카운트 등록
            else: # 다른 통로로 이동하는 경우 (교차통로 우회 3단계 맨해튼 궤적)
                add_line_segment(x1, y1, x1, CROSS_AISLE_Y)        # 1. 현재 랙 안쪽에서 교차로 지점(Y=3.0)까지 후퇴하는 직선 구간 카운트
                add_line_segment(x1, CROSS_AISLE_Y, x2, CROSS_AISLE_Y) # 2. 교차로 지점 내에서 목표 통로(X2)까지 가로로 횡단하는 구간 카운트
                add_line_segment(x2, CROSS_AISLE_Y, x2, y2)        # 3. 목표 통로 교차로에서 최종 랙 안쪽(Y2)으로 깊숙이 진입하는 직선 구간 카운트

    # 수집된 통행 빈도 데이터를 시각화를 위한 리스트로 분리
    pts_x, pts_y, pts_c = [], [], [] # X좌표 리스트, Y좌표 리스트, 빈도수(색상값) 리스트 초기화
    for (x, y), count in traffic_counts.items(): # 누적된 통행 딕셔너리를 순회
        pts_x.append(x) # 분리된 X 리스트에 추가
        pts_y.append(y) # 분리된 Y 리스트에 추가
        pts_c.append(count) # 빈도수를 리스트에 추가

    # -------------------------------------------------------------------
    # Scatter Plot을 활용해 선 대신 연속된 점들로 그라데이션(Heatmap) 렌더링
    # cmap='Reds': 한 번 지나가면 연한 핑크, 여러 번 지나가 병목인 곳은 진한 빨강
    # -------------------------------------------------------------------
    sc = ax2.scatter(pts_x, pts_y, c=pts_c, cmap='Reds', s=15, zorder=3, alpha=0.9) # 추출한 리스트를 산점도로 그리되 빈도(pts_c)에 따라 빨간색 농도를 다르게 적용
    
    # 우측에 컬러바를 부착하여 어떤 색상이 얼마나 많은 횟수를 의미하는지 범례 제공
    cbar = fig2.colorbar(sc, ax=ax2) # 방금 그린 산점도 색상 기준을 축으로 삼아 컬러바 생성
    cbar.set_label('로봇 누적 통행 빈도 (Traffic Intensity)') # 컬러바 옆 설명 텍스트 부착

    ax2.set_title(f'② 2D Trajectory Map: 로봇 통행 밀집도 궤적 ({slot_rule} × {disp_rule})', fontsize=14, fontweight='bold') # 2D 궤적 맵 제목에 현재 적용 중인 조합 변수명 동적 표시
    ax2.set_xlabel('X 좌표 (m)') # 궤적 맵 X축 물리적 단위(미터) 라벨
    ax2.set_ylabel('Y 좌표 (m)') # 궤적 맵 Y축 라벨
    ax2.set_xlim(-1, 15.5) # 맨 우측 랙이 잘리지 않도록 X축 오른쪽 여백 확대
    ax2.set_ylim(-1, 17) # 하단 스테이션 영역과 랙 상단 여백 표시를 위해 Y 범위 -1 ~ 17 확보
    
    # 기존 범례를 깔끔하게 유지하기 위한 정리 로직
    handles, labels = ax2.get_legend_handles_labels() # 플롯에 추가된 모든 막대 종류의 범례 핸들과 라벨 텍스트 수집
    by_label = dict(zip(labels, handles)) # 중복 라벨을 하나로 합치기 위해 딕셔너리 구조 활용
    ax2.legend(by_label.values(), by_label.keys(), loc='upper left') # 좌측 상단에 정돈된 범례 표시
    
    plt.tight_layout() # 간격 자동 조절
    plt.show() # 2D 궤적 맵 화면 출력

    # ---------------------------------------------------------
    # [3] Gantt Chart: CTU별 작업 시간 및 유휴 시간 확인
    # ---------------------------------------------------------
    fig3, ax3 = plt.subplots(figsize=(12, 4)) # 가로로 긴 형태(12x4)의 간트 차트 피규어 생성
    ctu_tasks = {0: [], 1: []} # 각 로봇의 (작업 시작 시점, 작업 소요 시간) 정보 튜플을 담을 빈 리스트 딕셔너리
    ctu_returns = {0: [], 1: []} # 복귀 구간도 별도의 색상으로 구분하기 위해 (시작, 소요) 튜플을 담을 딕셔너리
    
    for _, row in df_day.iterrows(): # 모든 이동 이력을 시간순으로 순회
        end_time = row['time'] # 해당 구간 이동이 끝난 완료 시간
        duration = row['travel_time'] # 해당 구간 이동에 걸린 소요 시간(초)
        start_time = end_time - duration # 이동 완료 시점에서 소요 시간을 빼 이동이 시작된 출발 시간 역산
        
        if row['item_id'] == 'RETURN': # 만약 이동 목적이 작업이 아닌 스테이션 복귀라면
            ctu_returns[row['ctu_id']].append((start_time, duration)) # 복귀 리스트에 시작, 지속시간 튜플 저장
        else: # 일반 부품 피킹 이동이라면
            ctu_tasks[row['ctu_id']].append((start_time, duration)) # 작업 리스트에 시작, 지속시간 저장
            
    ax3.broken_barh(ctu_tasks[0], (5, 8), facecolors='tab:blue', label='CTU 0 (피킹)') # CTU 0번 작업 리스트의 막대들을 Y축 기준 5~13 위치에 파란색으로 그림
    ax3.broken_barh(ctu_returns[0], (5, 8), facecolors='lightblue', label='CTU 0 (복귀)') # CTU 0번 복귀 구간 막대를 같은 Y축 라인에 연파랑으로 겹쳐서 표시
    ax3.broken_barh(ctu_tasks[1], (15, 8), facecolors='tab:red', label='CTU 1 (피킹)') # CTU 1번 작업 막대를 Y축 15~23 윗줄에 빨강으로 배치
    ax3.broken_barh(ctu_returns[1], (15, 8), facecolors='lightcoral', label='CTU 1 (복귀)') # CTU 1번 복귀 구간 연빨강 처리
    
    ax3.set_ylim(0, 30) # Y축의 범위를 0부터 30까지 넉넉하게 설정하여 두 줄의 막대가 중앙에 잘 보이도록 여백 확보
    ax3.set_xlim(0, df_day['time'].max() + 10) # X축(시간)을 0초부터 가장 마지막 기록된 종료 시간의 +10초 여유분까지 설정
    ax3.set_yticks([9, 19]) # Y축 좌측 라벨이 막대의 정중앙(5+4, 15+4)을 가리키도록 틱 위치 9, 19 지정
    ax3.set_yticklabels(['CTU 0', 'CTU 1']) # 지정된 틱 위치에 로봇 이름 텍스트 표시
    ax3.set_xlabel('시뮬레이션 시간 (초)') # X축이 시간 흐름임을 명시
    ax3.set_title(f'③ Gantt Chart: CTU 작업 흐름 ({slot_rule} × {disp_rule})', fontsize=14, fontweight='bold') # 간트 차트 제목에 조합 변수명 동적 표시
    
    handles, labels = ax3.get_legend_handles_labels() # 플롯에 추가된 모든 막대 종류의 범례 핸들과 라벨 텍스트 수집
    by_label = dict(zip(labels, handles)) # 중복 라벨을 하나로 합치기 위해 딕셔너리 구조 활용 (같은 이름이면 덮어씀)
    ax3.legend(by_label.values(), by_label.keys(), loc='upper right') # 중복 제거된 유니크한 범례를 우측 상단에 박스로 표시
    
    plt.tight_layout() # 차트 여백 최적화
    plt.show() # 최종 간트 차트 팝업 출력

# ==========================================
# 주 단위 런타임(소요 시간) 시각화 표 생성 함수 [수정]
# ==========================================
def report_weekly_scoreboard(res_df):
    """모든 조합의 일별 런타임을 주 단위로 합산하여 히트맵(Heatmap) 형태의 시각화 표로 출력합니다."""
    
    # 1. Date 컬럼을 datetime 타입으로 확실히 변환 (주차별 그룹화를 위해 필요)
    res_df['Date'] = pd.to_datetime(res_df['Date'])
    
    # ISO 캘린더 기준으로 연도와 주차(Week)를 추출하여 라벨용 파생 변수(Week) 생성 (예: '2026년 01주차')
    res_df['Week'] = res_df['Date'].dt.strftime('%G년 %V주차')
    
    # 시각화 표의 Y축(행)에 표시될 이름 구분을 위해 슬로팅 룰과 디스패칭 룰을 합친 조합(Combo) 파생 변수 생성
    res_df['Combo'] = res_df['Slotting'] + " + " + res_df['Dispatching']
    
    # 2. 주 단위 런타임 피벗 테이블 생성 
    # - index(행)에는 알고리즘 조합명을, columns(열)에는 계산된 주차 라벨을 배치
    # - values에는 기존 시뮬레이션 소요 시간(Time)을 넣되, aggfunc='sum'을 통해 해당 주차의 모든 일자 소요 시간을 합산
    weekly_pivot = res_df.pivot_table(index='Combo', columns='Week', values='Time', aggfunc='sum').round(1)
    
    # 3. 히트맵(Heatmap) 시각화 표 출력 영역 설정 (가로 14, 세로 8 크기)
    plt.figure(figsize=(14, 8))
    
    # Seaborn 히트맵 생성:
    # - annot=True를 통해 표의 각 셀 안에 실제 합산된 런타임 숫자(초)를 텍스트로 바로 표시
    # - fmt=".1f"를 통해 소수점 첫째 자리까지만 깔끔하게 출력
    # - cmap="YlOrRd" 설정으로 숫자가 클수록(오래 걸릴수록) 노랑 -> 주황 -> 빨강으로 직관적인 색상 변화 부여
    sns.heatmap(weekly_pivot, annot=True, fmt=".1f", cmap="YlOrRd", 
                linewidths=0.5, cbar_kws={'label': '주간 총 소요 시간 (초)'})
    
    # 시각화 표의 제목 및 축 레이블 설정
    plt.title('알고리즘 조합별 주간 시뮬레이션 총 소요 시간 (단위: 초)', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('생산 주차 (Week)', fontsize=12)
    plt.ylabel('알고리즘 조합 (Slotting + Dispatching)', fontsize=12)
    
    # Y축 라벨(알고리즘 조합명) 글자가 기본적으로 세로로 누워있는 것을 막기 위해 가로(0도)로 회전
    plt.yticks(rotation=0) 
    
    # 범례와 표가 겹치지 않도록 전체적인 간격 자동 최적화
    plt.tight_layout()
    # 화면에 실제 시각화 표 팝업 출력
    plt.show()

# ==========================================
# main 함수 (수정)
# ==========================================
def main():
    # 1. 데이터 파일 읽기 (주문 계획, 생산 계획, 부품 BOM 구조)
    outbound_plan = pd.read_csv('outbound_plan.csv')
    production_plan = pd.read_csv('production_plan.csv')
    bom_df = pd.read_excel('BOM.xlsx')

    # 일자별 그룹화를 위해 날짜 데이터를 문자열이나 타임스탬프에서 순수 Date(날짜) 객체로 통일
    outbound_plan['생산일자'] = pd.to_datetime(outbound_plan['생산일자']).dt.date
    production_plan['생산일자'] = pd.to_datetime(production_plan['생산일자']).dt.date
    
    # 시뮬레이션을 돌려야 할 고유한 생산 일자 리스트를 정렬하여 추출
    unique_dates = sorted(outbound_plan['생산일자'].unique())
    # 출고 계획에 존재하는 모든 종류의 부품 코드 목록 확보
    part_list = outbound_plan['부품코드'].unique().tolist() 
    
    # 3. 실험할 알고리즘 조합 정의 (총 4개 슬로팅 * 3개 디스패칭 = 12개 조합 생성)
    slotting_rules = ['random', 'abc', 'characteristic', 'kitting']
    dispatch_rules = ['nearest', 'density', 'load_balance']

    # 시뮬레이션 결과를 누적 저장할 빈 리스트들 초기화
    all_results_summary = []
    move_logs_all = []
    dispatch_logs_all = []

    print(f"총 {len(unique_dates)}일치 데이터에 대해 시뮬레이션을 시작합니다. (총 12개 조합)")

    # 4. 시뮬레이션 실행 2중 루프 (슬로팅 룰 4종을 순회하며 내부에 디스패칭 룰 3종 반복)
    for s_rule in slotting_rules:
        for d_rule in dispatch_rules:
            inventory = None # 처음 시작 시 재고 데이터는 None으로 초기화
            
            # 모든 시뮬레이션 일자를 순차적으로 실행
            for i, date in enumerate(unique_dates):
                # [핵심] 7일마다 창고 내부 재배치(Slotting) 갱신 로직
                # i가 0, 7, 14 ... 일 때 즉 매주의 첫날에만 새로운 재고 배치를 수행
                if i % 7 == 0:
                    # 앞으로 다가올 7일치 일자 목록 추출
                    upcoming_7 = unique_dates[i : i+7]
                    # 생산 계획 테이블에서 향후 7일간의 물량 데이터를 필터링하여 주간 계획(weekly_plan) 확보
                    weekly_plan = production_plan[production_plan['생산일자'].isin(upcoming_7)]
                    
                    # 해당 주간 계획과 선택된 슬로팅 룰(s_rule)을 기반으로 새로운 창고 배치도(inventory) 생성
                    inventory = generate_fixed_inventory(s_rule, part_list, weekly_plan, bom_df, outbound_plan, seed=100)
                
                    print(f">> {date} [{s_rule} x {d_rule}] 실행 중...")

                # 해당 날짜에 처리해야 할 당일치 출고 계획(주문서) 필터링
                date_plan = outbound_plan[outbound_plan['생산일자'] == date]
                
                # 시뮬레이터 구동: 특정 날짜, 재고 배치도, 디스패칭 룰, 당일 주문서를 인자로 넘겨 하루 동안의 로봇 작업 시뮬레이션 실행
                res = run_simulation(date, inventory, d_rule, date_plan)

                # 하루치 시뮬레이션 결과 딕셔너리에 현재 적용되었던 룰 메타데이터 추가
                res['Slotting'] = s_rule
                res['Dispatching'] = d_rule
                
                # 요약 결과 리스트에 하루치 성적 저장
                all_results_summary.append(res)

                # 이동 이력 로그에도 룰 메타데이터를 추가한 뒤 전체 로그 풀에 병합
                move_tmp = res['Move_log'].copy()
                move_tmp['Slotting'], move_tmp['Dispatching'], move_tmp['Date'] = s_rule, d_rule, date
                move_logs_all.append(move_tmp)

                # 디스패칭(배치 할당) 로그에도 동일하게 메타데이터 추가 후 병합
                disp_tmp = res['Dispatch_log'].copy()
                disp_tmp['Slotting'], disp_tmp['Dispatching'], disp_tmp['Date'] = s_rule, d_rule, date
                dispatch_logs_all.append(disp_tmp)

    # 5. 최종 분석 및 모든 조합 시각화
    # 수집된 하루 단위 요약 결과들을 하나의 묶음 데이터프레임으로 변환
    df_res = pd.DataFrame(all_results_summary)
    # 수집된 모든 로봇 이동 로그를 하나의 거대한 묶음 데이터프레임으로 통합 (인덱스 리셋)
    move_all_df = pd.concat(move_logs_all, ignore_index=True)
    
    # (1) 일별 텍스트 보드 대신, 주 단위 히트맵 시각화 표 (12개 조합) 출력 함수 호출
    report_weekly_scoreboard(df_res)
    
    print("\n" + "="*60)
    print(" 모든 조합(12개)에 대한 상세 리포트 그래프를 생성합니다.")
    print(" ※ 주의: 총 12개의 조합 시각화 창이 순차적으로 열립니다.")
    print("="*60)

    # (2) 12개 조합에 대한 개별 3D, 2D, Gantt 시각화 함수 순차적 호출
    for s_rule in slotting_rules:
        for d_rule in dispatch_rules:
            visualize_rule_combination(move_all_df, s_rule, d_rule)

if __name__ == "__main__":
    main()